# HomeLens AI — NLP-Based Risk Intelligence for Home-Service Providers
**CIS 509 | Analytics for Unstructured Data — Final Course Project**

**Team HomeLens AI:** Rithik Roy Thati · Kanha Jodhpurkar · Sankalp Sharma Madgula · Dev Bhattacharyya

*Turning free-form customer reviews into recurring quality, reliability, pricing, and safety risk signals.*

---

### How to read and run this notebook

| Section | What it does | Rubric criterion |
|---|---|---|
| 1 | Business problem, stakeholders, research questions | Business Problem Definition |
| 2 | Setup, configuration, helper code | Code Clarity & Quality |
| 3 | Load, filter and clean the Yelp home-service reviews | EDA (data cleaning) |
| 4 | Exploratory data analysis | Exploratory Data Analysis |
| 5 | **Layer 1 — Sentiment:** VADER vs TF-IDF + Logistic Regression vs DistilBERT | NLP Methodology |
| 6 | **Layer 2 — Aspect extraction:** spaCy `PhraseMatcher`, `Matcher`, `EntityRuler` | NLP Methodology |
| 7 | **Layer 3 — Provider risk intelligence:** recurring issues, trend, risk tier, action | Results & Business Insights |
| 8 | Manager-facing risk dashboard and key insights | Results & Business Insights |
| 9 | Export artifacts for the web app (backend / frontend) | — |
| 10 | Conclusions, limitations, next steps | Results & Business Insights |

**Data.** Put the Yelp download in `data/raw/` — the **zip files exactly as downloaded (no unzipping needed)**, the two raw JSON files, or the pre-extracted `HomeLens_Yelp_HomeServices.csv` — see `data/README.md`.
If no real data is found the notebook **automatically falls back to a small synthetic sample** (`data/sample/sample_reviews.csv`) and prints a
loud *DEMO MODE* banner, so the whole pipeline can be exercised by teammates and CI. **Numbers in DEMO MODE are not results.**

**Run time (real data, ~16k reviews).** Everything except DistilBERT runs in a few minutes on a laptop CPU. DistilBERT fine-tuning wants a GPU
(NVIDIA CUDA or Apple-silicon MPS, ≈10–20 min); without one it is skipped automatically and the rest of the notebook is unaffected (`HOMELENS_RUN_DISTILBERT=1` forces it on CPU).

---
## 1. Business Problem Definition

### 1.1 The problem
Home services is a **high-cost, low-frequency, high-trust** category. A homeowner hires a roofer once a decade and has almost no basis for judging quality;
a service manager oversees dozens of technicians and only learns about problems *after* the star rating drops. Both groups work from star averages, which say
that something went wrong but not **what** went wrong, **whether it keeps happening**, or **what to do about it**. The signal that would answer those questions is
buried in thousands of free-form reviews that nobody has time to read.

> **Business question.** Can NLP identify recurring *operational, quality, pricing, communication, workmanship, warranty and safety* risk signals in home-service
> reviews, and help managers and homeowners prioritise the ones that matter?

### 1.2 Why this is a data-driven problem worth solving
* **Stars hide the cause.** Two providers with a 3.8 average can fail for completely different reasons (one is unreliable, the other is a pricing risk); the remedy differs.
* **Stars lag.** An average over 10 years barely moves when a provider's last 12 months get worse; recent-versus-prior text signals can surface this earlier.
* **Severity is not symmetric.** A review describing a gas leak or exposed wiring matters far more than one about a late arrival, yet both are "1 star".
* **Volume.** Reading every review is infeasible; ranking providers by *recurring, severity-weighted* complaint signals turns thousands of reviews into a short, prioritised worklist.

### 1.3 Stakeholders and the decision each one makes
| Stakeholder | Decision supported by HomeLens AI |
|---|---|
| Owners / operations managers | Which crews or issue types generate repeat complaints → where to coach, audit or intervene first |
| Multi-location leadership | Which locations are trending worse → where to allocate quality-control resources |
| Reputation / customer-care teams | Which reviews demand a response today (safety, warranty) versus routine acknowledgement |
| Homeowners | Which providers carry lower recurring risk for the *specific* thing they care about (price, punctuality, safety) |

### 1.4 Research questions
* **RQ1 (Sentiment).** How well can a review's text alone recover a "risk signal" label (1–2★) versus "no immediate risk" (4–5★), and does a fine-tuned transformer
  beat a lexicon baseline and a classical TF-IDF model — especially on **negative-class recall**, where a miss is costlier than a false alarm?
* **RQ2 (Aspects).** Can transparent, rule-based spaCy matchers tag the *type* of complaint (workmanship, reliability, pricing, communication, timeliness,
  professionalism, warranty, safety) and return the **evidence sentence** so that every flag is explainable?
* **RQ3 (Provider intelligence).** Do provider-level roll-ups (recurrence, recent-vs-prior trend, severity) surface risks that the provider's star average hides?
* **RQ4 (Mixed reviews).** What do 3★ reviews — deliberately held out of training — look like to the model and to the aspect rules?

### 1.5 Success criteria
| Layer | Metric | Target |
|---|---|---|
| 1 – Sentiment | Negative-class (risk) **recall**, plus macro / weighted F1 | Recall ≥ 0.90 at the chosen operating point; transformer ≥ TF-IDF ≥ VADER on macro-F1 |
| 2 – Aspects | Precision of flagged evidence sentences (manual audit); rule unit-tests; lift in 1–2★ vs 4–5★ reviews | Audit precision ≥ 0.80; every aspect shows lift > 1 |
| 3 – Providers | Providers with ≥ 20 reviews get a score, tier, trend, top issues and recommended action | Surfaces providers with *good stars but recurring issues* |

### 1.6 A caution that shapes the design
Reviews are **customer allegations, not verified facts**. The system therefore reports a *"complaint signal detected"* — never a finding of wrongdoing — and always
shows the underlying evidence sentence so a human can judge it.

### 1.7 Approach: three layers
```
 Yelp reviews (free text) ──► Layer 1  SENTIMENT      "Is this review a risk signal?"      VADER │ TF-IDF+LR │ DistilBERT
                          ──► Layer 2  ASPECTS        "What went wrong? Show me the proof"  spaCy PhraseMatcher + Matcher + EntityRuler
                          ──► Layer 3  PROVIDER RISK  "Does it keep happening? What now?"   recurrence · trend · severity → tier → action
                                          │
                                          ▼
                                 Manager-facing risk dashboard  (+ JSON artifacts for the web app)
```
*Scope note.* Our original proposal framed HomeLens AI as multimodal home-price valuation. After auditing the data and the Milestone 2 requirements we refined the
target to home-service provider quality and risk intelligence; the team name and product concept carry forward.

---
## 2. Setup and Configuration

In [ ]:
# --- Install (run once in a fresh environment): `pip install -r requirements.txt` ----------------------------------------------
# !pip install -r requirements.txt && python -m spacy download en_core_web_sm
# !pip install -r requirements-bert.txt        # only for the DistilBERT section (a GPU is strongly recommended)

In [ ]:
import json
import math
import os
import random
import re
import sys
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold", "axes.titlesize": 12})

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# ============================== CONFIGURATION — edit here, nowhere else ==================================================
SEED = 509                                  # one seed for every random operation → reproducible results

# ---- Data ----------------------------------------------------------------------------------------------------------------
EXTRACT_NAME = "HomeLens_Yelp_HomeServices.csv"   # pre-extracted Tucson, AZ home-service reviews (Milestone 1)
CITY, STATE = "Tucson", "AZ"                      # used only when building the extract from the raw Yelp JSON files
MIN_WORDS = 3                                     # drop reviews shorter than this (no usable text)

# ---- Layer 1: sentiment --------------------------------------------------------------------------------------------------
RISK_STARS, SAFE_STARS, MIXED_STARS = (1, 2), (4, 5), (3,)   # label rule from the proposal; 3★ held out
TEST_SIZE, VAL_SIZE = 0.15, 0.15
TARGET_RISK_RECALL = 0.90                         # business constraint: a missed serious complaint costs more than a false alarm
F_BETA = 2.0                                      # recall-weighted F-score used to pick the decision threshold
RUN_DISTILBERT = {"1": True, "0": False}.get(os.environ.get("HOMELENS_RUN_DISTILBERT", ""), "auto")   # True | False | "auto" (= only with a CUDA GPU)
BERT_CHECKPOINT = os.environ.get("HOMELENS_BERT_CHECKPOINT", "distilbert-base-uncased")
BERT_MAX_LEN, BERT_EPOCHS, BERT_BATCH, BERT_LR = 256, 3, 32, 3e-5
BERT_MAX_TRAIN = None                             # e.g. 3000 to cap training-set size for a quick CPU smoke test

# ---- Layer 2: aspects ----------------------------------------------------------------------------------------------------
NEG_SENT_THRESH = -0.05                           # VADER sentence compound at/below which an aspect cue counts as a complaint

# ---- Layer 3: provider risk ----------------------------------------------------------------------------------------------
MIN_REVIEWS_PER_PROVIDER = 20                     # minimum for a reliable ranking (proposal)
RECENT_MONTHS = 24                                # "recent" window; the "prior" window is the RECENT_MONTHS before it
MIN_WINDOW_REVIEWS = 5                            # min reviews in BOTH windows to call a trend
TREND_ALPHA = 0.10                                # Fisher exact test level for worsening / improving
RECURRENCE_MIN = 3                                # an issue is "recurring" when ≥ this many distinct reviews mention it
TIER_PCTS = {"High": 0.85, "Medium": 0.60}        # score percentiles (among eligible providers) that start each tier

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ---- Locate the project folders (works from the repo root or from notebooks/ ---------------------------
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return here                                   # fall back to the current working directory

REPO_ROOT = find_repo_root()
DATA_DIR = Path(os.environ.get("HOMELENS_DATA_DIR", REPO_ROOT / "data"))
RAW_DIR = DATA_DIR / "raw"
SAMPLE_PATH = DATA_DIR / "sample" / "sample_reviews.csv"


ARCHIVE_EXTS = (".zip", ".tar", ".tar.gz", ".tgz")


def _archives():
    return sorted(p for p in RAW_DIR.rglob("*") if p.is_file() and p.name.lower().endswith(ARCHIVE_EXTS)) if RAW_DIR.exists() else []


def locate_data():
    """Return (mode, source). Preference: pre-extracted CSV > raw Yelp JSON (plain files OR zip/tar archives, no unzipping needed) > synthetic sample."""
    for p in (RAW_DIR / EXTRACT_NAME, DATA_DIR / EXTRACT_NAME):
        if p.exists():
            return "extract", p
    plain = [p for p in RAW_DIR.rglob("yelp_academic_dataset_*.json")] if RAW_DIR.exists() else []
    if plain or _archives():
        return "raw_json", (plain or _archives())
    if SAMPLE_PATH.exists():
        return "sample", SAMPLE_PATH
    raise FileNotFoundError(
        f"No data found. Put the Yelp zip files (or {EXTRACT_NAME}) in {RAW_DIR} — see data/README.md — or run `python scripts/make_sample_data.py`."
    )


DATA_MODE, DATA_SRC = locate_data()
DEMO_MODE = DATA_MODE == "sample"
OUTPUT_DIR = (REPO_ROOT / "data" / "sample" / "outputs") if DEMO_MODE else (DATA_DIR / "processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root : {REPO_ROOT}\ndata mode : {DATA_MODE}\ndata src  : {DATA_SRC}\noutputs   : {OUTPUT_DIR}")
if DEMO_MODE:
    display(Markdown(
        "> ## ⚠️ DEMO MODE\n> The real Yelp data was not found, so this run uses a **synthetic** sample. "
        "All numbers, charts and rankings below are for pipeline testing only and are **not project results**."
    ))

---
## 3. Data: Loading, Home-Service Filtering and Cleaning

**Sources (Yelp Open Dataset).** The primary unstructured field is the free-form review `text`. Structured business fields (`name`, `categories`, `city`,
business-level `stars`, `review_count`) are used only for context, filtering and provider-level aggregation, joined on `business_id`.

The Milestone 1 extract contains 23,685 reviews from 1,403 Tucson, AZ home-service businesses (2006–2022). Many of those are *not* service providers
(furniture stores, real-estate agencies, hotels, nurseries …), so we narrow to **true service providers** with an explicit, auditable category rule.

In [ ]:
# ---- Category rules: which businesses are *service providers* (contractors, plumbers, movers …) and which trade are they? ------
# Matching is on lower-cased category text. Order matters: the first matching trade wins (specific trades before generic ones).
TRADE_RULES = {
    "Plumbing": ["plumb", "water heater", "drain", "septic", "sewer", "water purification"],
    "Electrical": ["electrician", "electrical"],
    "HVAC": ["heating & air", "hvac", "air conditioning", "furnace", "air duct"],
    "Roofing": ["roofing", "gutter"],
    "Movers": ["movers", "packing services", "moving"],
    "Cleaning": ["home cleaning", "carpet cleaning", "janitorial", "window washing", "pressure washing", "junk removal", "upholstery"],
    "Landscaping & Outdoor": ["landscap", "tree service", "lawn", "gardener", "irrigation", "pest control", "pool & hot tub", "pool clean", "fences"],
    "Contracting & Remodeling": [
        "contractors", "handyman", "remodel", "carpent", "masonry", "concrete", "drywall", "siding", "flooring", "tiling", "insulation",
        "framing", "windows installation", "door sales", "garage door", "decks & railing", "patio", "solar installation", "painters",
        "countertop", "cabinetry", "building supplies installation",
    ],
    "Other Home Services": ["appliances & repair", "locksmith", "home inspector", "security systems", "chimney", "glass & mirrors", "TV mounting"],
}
# Exact category tokens that mark a *retailer / lodging / real-estate* listing. These are removed even if a service term also matches.
NON_PROVIDER_TOKENS = {
    "hardware stores", "furniture stores", "mattresses", "building supplies", "home decor", "department stores", "nurseries & gardening",
    "appliances", "kitchen & bath", "real estate", "real estate services", "real estate agents", "apartments", "hotels", "property management",
    "home staging", "interior design", "rugs", "lumber", "hotels & travel",
}


def split_categories(cat: str) -> list[str]:
    return [c.strip() for c in str(cat).split(",") if c.strip()]


def assign_trade(categories: str) -> str | None:
    """Return the trade label for a business, or None if it is not a true service provider."""
    tokens = {c.lower() for c in split_categories(categories)}
    if tokens & NON_PROVIDER_TOKENS:
        return None
    text = " | ".join(sorted(tokens))
    for trade, needles in TRADE_RULES.items():
        if any(n.lower() in text for n in needles):
            return trade
    return None


# Fast self-check so a teammate who edits the rules sees immediately if something breaks
assert assign_trade("Home Services, Plumbing, Water Heater Installation/Repair") == "Plumbing"
assert assign_trade("Home Services, Roofing, Contractors") == "Roofing"
assert assign_trade("Furniture Stores, Home & Garden, Home Decor") is None
assert assign_trade("Real Estate, Home Services, Contractors") is None
print("category rules OK")

In [ ]:
# ---- Loading helpers ------------------------------------------------------------------------------------------------------------
COLUMN_ALIASES = {
    "review_id": ["review_id"],
    "business_id": ["business_id"],
    "text": ["text", "review_text", "review"],
    "date": ["date", "review_date"],
    "business_name": ["business_name", "name", "name_business", "name_y", "name_x"],
    "categories": ["categories"],
    "city": ["city"],
    "state": ["state"],
    "business_stars": ["business_stars", "stars_business", "avg_stars"],
    "business_review_count": ["business_review_count", "review_count_business", "review_count_y", "review_count"],
}


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Map whatever the extract calls its columns onto one canonical schema; tolerate the stars_x / stars_y merge suffixes."""
    df = df.copy()
    rename = {}
    for canon, options in COLUMN_ALIASES.items():
        for opt in options:
            if opt in df.columns and canon not in df.columns and opt not in rename:
                rename[opt] = canon
                break
    df = df.rename(columns=rename)

    if "review_stars" not in df.columns:
        # Review stars are whole numbers 1-5; business stars come in 0.5 steps. Use that to tell the two apart after a merge.
        candidates = [c for c in ("stars_x", "stars_y", "stars", "stars_review", "rating") if c in df.columns]
        whole = [c for c in candidates if pd.to_numeric(df[c], errors="coerce").dropna().mod(1).eq(0).all()]
        if not whole:
            raise KeyError(f"Cannot find the review-level star column among {list(df.columns)}")
        df = df.rename(columns={whole[0]: "review_stars"})
        other = [c for c in candidates if c != whole[0]]
        if "business_stars" not in df.columns and other:
            df = df.rename(columns={other[0]: "business_stars"})
    missing = [c for c in ("business_id", "text", "date", "review_stars", "categories") if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns after standardisation: {missing}")
    for optional in ("review_id", "business_name", "city", "state", "business_stars", "business_review_count"):
        if optional not in df.columns:
            df[optional] = np.nan
    if df["review_id"].isna().all():
        df["review_id"] = [f"rev_{i:06d}" for i in range(len(df))]
    return df


import contextlib
import io
import tarfile
import zipfile


def _search_archive(src, name, target, stack, depth=0):
    """Depth-first search of a zip / tar(.gz) (and archives nested inside) for a member called `target`.
    Returns a binary stream or None. Streams where possible — a 5 GB JSON is never unpacked to disk unless it sits in a *zip inside a zip*."""
    lower = name.lower()
    if lower.endswith(".zip"):
        zf = stack.enter_context(zipfile.ZipFile(src))
        for info in zf.infolist():
            if Path(info.filename).name == target:
                return stack.enter_context(zf.open(info))
        if depth < 3:
            for info in zf.infolist():
                inner = info.filename.lower()
                if inner.endswith((".tar", ".tar.gz", ".tgz")):
                    found = _search_archive(stack.enter_context(zf.open(info)), Path(info.filename).name, target, stack, depth + 1)
                    if found is not None:
                        return found
                elif inner.endswith(".zip") and "__macosx" not in inner:
                    unpacked = CACHE_DIR / "unpacked" / Path(info.filename).name        # zip-in-zip needs a seekable file
                    if not unpacked.exists():
                        unpacked.parent.mkdir(parents=True, exist_ok=True)
                        print(f"  unpacking nested archive {info.filename} → {unpacked} (one-off) …")
                        with zf.open(info) as fin, open(unpacked, "wb") as fout:
                            while chunk := fin.read(8 << 20):
                                fout.write(chunk)
                    found = _search_archive(unpacked, unpacked.name, target, stack, depth + 1)
                    if found is not None:
                        return found
    elif lower.endswith((".tar", ".tar.gz", ".tgz")):
        fileobj = src if hasattr(src, "read") else stack.enter_context(open(src, "rb"))
        tf = stack.enter_context(tarfile.open(fileobj=fileobj, mode="r|*"))            # streaming mode: no seeking, works inside a zip
        for member in tf:
            if member.isfile() and Path(member.name).name == target:
                return tf.extractfile(member)
    return None


class _ReadOnlyStream(io.RawIOBase):
    """Adapter so any binary stream (zip member, streamed tar member) can sit under io.TextIOWrapper / pandas."""

    def __init__(self, inner):
        self.inner = inner

    def readable(self):
        return True

    def seekable(self):
        return False

    def readinto(self, buf):
        data = self.inner.read(len(buf))
        buf[: len(data)] = data
        return len(data)


@contextlib.contextmanager
def open_yelp_json(kind: str):
    """Yield a text stream of yelp_academic_dataset_<kind>.json from a plain file, a zip, a tar, or an archive inside an archive."""
    target = f"yelp_academic_dataset_{kind}.json"
    with contextlib.ExitStack() as stack:
        for p in RAW_DIR.rglob(target):
            yield stack.enter_context(open(p, encoding="utf-8"))
            return
        for arc in _archives():
            stream = _search_archive(arc, arc.name, target, stack)
            if stream is not None:
                print(f"  reading {target} straight from {arc.name}")
                yield io.TextIOWrapper(io.BufferedReader(_ReadOnlyStream(stream), buffer_size=8 << 20), encoding="utf-8")
                return
    raise FileNotFoundError(f"{target} not found in {RAW_DIR} (looked in plain files and in {[a.name for a in _archives()]}).")


def build_extract_from_raw() -> pd.DataFrame:
    """Stream the (multi-GB) raw Yelp files once and keep only home-service-related reviews for CITY, STATE."""
    with open_yelp_json("business") as f:
        biz = pd.read_json(f, lines=True)
    biz = biz[(biz["city"].str.lower() == CITY.lower()) & (biz["state"] == STATE) & biz["categories"].notna()]
    coarse = biz["categories"].str.contains("Home Services|Contractors", case=False, regex=True)
    biz = biz[coarse].rename(columns={"name": "business_name", "stars": "business_stars", "review_count": "business_review_count"})
    keep = set(biz["business_id"])
    print(f"  {len(keep):,} candidate home-service businesses in {CITY}, {STATE}; scanning reviews …")
    parts, seen = [], 0
    with open_yelp_json("review") as f:
        for chunk in pd.read_json(f, lines=True, chunksize=200_000):
            seen += len(chunk)
            parts.append(chunk[chunk["business_id"].isin(keep)])
            print(f"  scanned {seen:,} reviews, kept {sum(map(len, parts)):,}", end="\r")
    print()
    rev = pd.concat(parts, ignore_index=True).rename(columns={"stars": "review_stars"})
    return rev.merge(biz[["business_id", "business_name", "categories", "city", "state", "business_stars", "business_review_count"]], on="business_id", how="inner")


def load_raw_reviews() -> pd.DataFrame:
    if DATA_MODE == "raw_json":
        print("Building the extract from the raw Yelp files (one-off, a few minutes) …")
        df = build_extract_from_raw()
        out = RAW_DIR / EXTRACT_NAME
        df.to_csv(out, index=False)
        print(f"saved {len(df):,} reviews → {out}")
        return df
    return pd.read_csv(DATA_SRC)


raw = standardize_columns(load_raw_reviews())
print(f"loaded {len(raw):,} reviews from {raw['business_id'].nunique():,} businesses")
raw.head(3)

In [ ]:
# ---- Filter to true service providers + clean text, with a log of every step (an audit trail for the report) -----------------
cleaning_log = [("Loaded extract", len(raw), raw["business_id"].nunique())]


def log_step(name: str, frame: pd.DataFrame):
    cleaning_log.append((name, len(frame), frame["business_id"].nunique()))


df = raw.copy()

# 1. keep true service providers (and record what we excluded so the rule is auditable)
df["trade"] = df["categories"].map(assign_trade)
excluded_cats = (
    df[df["trade"].isna()].drop_duplicates("business_id")["categories"].map(split_categories).explode().value_counts().head(15)
)
df = df[df["trade"].notna()]
log_step("Keep true service providers (category rule)", df)

# 2. basic validity
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["review_stars"] = pd.to_numeric(df["review_stars"], errors="coerce")
df = df.dropna(subset=["business_id", "text", "date", "review_stars"])
df = df[df["review_stars"].between(1, 5)]
df["review_stars"] = df["review_stars"].astype(int)
log_step("Drop missing text / date / stars", df)

# 3. text normalisation: HTML entities, URLs, control characters, runs of whitespace — wording itself is left untouched
def clean_text(t: str) -> str:
    t = str(t).replace("&amp;", "&").replace("&quot;", '"').replace("&#39;", "'").replace("\\n", " ")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


df["text"] = df["text"].map(clean_text)
df["n_words"] = df["text"].str.split().str.len()
df = df[df["n_words"] >= MIN_WORDS]
log_step(f"Drop reviews shorter than {MIN_WORDS} words", df)

# 4. duplicates: same review id, or identical text for the same business (copy-pasted / double-posted reviews)
df = df.drop_duplicates("review_id").drop_duplicates(["business_id", "text"])
log_step("Drop duplicate reviews", df)

df = df.sort_values("date").reset_index(drop=True)
df["year"] = df["date"].dt.year
df["n_chars"] = df["text"].str.len()

cleaning_log = pd.DataFrame(cleaning_log, columns=["step", "reviews", "providers"])
cleaning_log["reviews_removed"] = (-cleaning_log["reviews"].diff()).fillna(0).astype(int)
display(cleaning_log)
print(f"\nAnalysis set: {len(df):,} reviews | {df['business_id'].nunique():,} providers | {df['date'].min():%Y-%m-%d} → {df['date'].max():%Y-%m-%d}")
print("Most common categories on EXCLUDED businesses (sanity check of the filter — adjust TRADE_RULES if something real is here):")
print(excluded_cats.to_string())

---
## 4. Exploratory Data Analysis

The EDA answers four practical questions that drive later modelling decisions:
1. **How balanced are the labels?** (decides class weighting and the evaluation metric)
2. **How long are the reviews?** (decides the transformer's max sequence length)
3. **How is the data distributed over providers and time?** (decides the 20-review minimum and the recent-vs-prior window)
4. **Which words separate complaints from praise?** (a sanity check that the text carries the signal we want to model)

Every chart is followed by a *computed* takeaway, so the text always matches the data being analysed.

In [ ]:
# ---- 4.1 Key statistics --------------------------------------------------------------------------------------------------------
per_provider = df.groupby("business_id").size()
tokens_total = int(df["n_words"].sum())
key_stats = pd.Series({
    "Reviews (analysis set)": f"{len(df):,}",
    "Service providers": f"{df['business_id'].nunique():,}",
    "Trades": f"{df['trade'].nunique()}",
    "Date range": f"{df['date'].min():%b %Y} – {df['date'].max():%b %Y}",
    "Total words": f"{tokens_total:,}",
    "Median words / review": f"{df['n_words'].median():.0f}",
    "Mean words / review": f"{df['n_words'].mean():.1f}",
    "95th pct words / review": f"{df['n_words'].quantile(.95):.0f}",
    "Mean review stars": f"{df['review_stars'].mean():.2f}",
    "Providers with ≥ MIN_REVIEWS": f"{(per_provider >= MIN_REVIEWS_PER_PROVIDER).sum():,}",
    "Median reviews / provider": f"{per_provider.median():.0f}",
}, name="value").to_frame()
display(key_stats)

display(df[["review_stars", "n_words", "n_chars"]].describe().T)
print("Missing values in key fields:\n", df[["business_id", "text", "date", "review_stars", "categories"]].isna().sum().to_string())

In [ ]:
# ---- 4.2 Label balance, review length, volume over time, provider long-tail ------------------------------------------------
star_counts = df["review_stars"].value_counts().sort_index()
group = df["review_stars"].map(lambda s: "Risk signal (1–2★)" if s in RISK_STARS else "Mixed (3★)" if s in MIXED_STARS else "No immediate risk (4–5★)")
palette = {"Risk signal (1–2★)": "#d1495b", "Mixed (3★)": "#edae49", "No immediate risk (4–5★)": "#00798c"}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
colors = ["#d1495b", "#d1495b", "#edae49", "#00798c", "#00798c"]
ax.bar(star_counts.index, star_counts.values, color=colors)
for x, y in star_counts.items():
    ax.text(x, y, f"{y / len(df):.0%}", ha="center", va="bottom", fontsize=9)
ax.set(title="Review star distribution", xlabel="Review stars", ylabel="Reviews")

ax = axes[0, 1]
for g, c in palette.items():
    ax.hist(np.log10(df.loc[group == g, "n_words"].clip(lower=1)), bins=40, alpha=0.55, color=c, label=g, density=True)
ax.set_xticks([1, 1.5, 2, 2.5, 3], ["10", "32", "100", "316", "1000"])
ax.set(title="Review length by label group (words, log scale)", xlabel="Words per review", ylabel="Density")
ax.legend(fontsize=8)

ax = axes[1, 0]
yearly = df.groupby("year").agg(reviews=("review_id", "size"), mean_stars=("review_stars", "mean"))
ax.bar(yearly.index, yearly["reviews"], color="#9aa5b1")
ax.set(title="Reviews per year (bars) and mean stars (line)", xlabel="Year", ylabel="Reviews")
ax2 = ax.twinx()
ax2.plot(yearly.index, yearly["mean_stars"], color="#d1495b", marker="o")
ax2.set(ylabel="Mean stars", ylim=(1, 5))
ax2.grid(False)

ax = axes[1, 1]
ax.hist(per_provider.clip(upper=100), bins=50, color="#6a8caf")
ax.axvline(MIN_REVIEWS_PER_PROVIDER, color="#d1495b", ls="--", label=f"min {MIN_REVIEWS_PER_PROVIDER} reviews")
ax.set(title="Reviews per provider (long tail, clipped at 100)", xlabel="Reviews", ylabel="Providers")
ax.legend()
plt.tight_layout()
plt.show()

risk_share, safe_share = df["review_stars"].isin(RISK_STARS).mean(), df["review_stars"].isin(SAFE_STARS).mean()
len_by_grp = df.groupby(group)["n_words"].median()
eligible = per_provider[per_provider >= MIN_REVIEWS_PER_PROVIDER]
len_risk, len_safe = len_by_grp.get("Risk signal (1–2★)", np.nan), len_by_grp.get("No immediate risk (4–5★)", np.nan)
len_msg = (
    f"risk-signal reviews are {'longer' if len_risk > len_safe else 'shorter'} (median {len_risk:.0f} words) than 4–5★ reviews ({len_safe:.0f})"
    + (" — dissatisfied customers explain more, which is exactly the text worth mining" if len_risk > len_safe else "")
)
last5 = (df["year"] >= df["year"].max() - 4).mean()
display(Markdown(
    f"**Takeaways.** "
    f"**(1) Imbalance:** {risk_share:.0%} of reviews are 1–2★ (risk signal) versus {safe_share:.0%} 4–5★, so accuracy is misleading "
    f"(always predicting 'no risk' already scores {safe_share / (risk_share + safe_share):.0%}); we therefore use class weights and report **macro-F1 and negative-class recall**. "
    f"**(2) Length:** the median review is {df['n_words'].median():.0f} words and the 95th percentile {df['n_words'].quantile(.95):.0f}; {len_msg}. "
    f"A {BERT_MAX_LEN}-token window covers {(df['n_words'] * 1.3 <= BERT_MAX_LEN).mean():.0%} of reviews without truncation (≈1.3 tokens per word). "
    f"**(3) Coverage:** {len(eligible):,} of {len(per_provider):,} providers ({len(eligible) / len(per_provider):.0%}) reach {MIN_REVIEWS_PER_PROVIDER} reviews and they hold "
    f"{eligible.sum() / len(df):.0%} of all reviews, so the minimum-volume rule removes the least reliable providers while keeping most of the evidence. "
    f"**(4) Time:** {last5:.0%} of reviews fall in the last five years of the data, which determines how many providers have enough reviews in both the recent and prior windows used for trend detection."
))

In [ ]:
# ---- 4.3 Trades: how much data and how much risk signal per trade? ---------------------------------------------------------------
def wilson_interval(k, n, z=1.96):
    """Wilson score interval for a proportion — well-behaved for small n and for rates near 0 or 1."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


trade_tbl = (
    df.assign(is_risk=df["review_stars"].isin(RISK_STARS))
    .groupby("trade")
    .agg(providers=("business_id", "nunique"), reviews=("review_id", "size"), mean_stars=("review_stars", "mean"), risk_reviews=("is_risk", "sum"))
    .sort_values("reviews", ascending=False)
)
trade_tbl["risk_rate"] = trade_tbl["risk_reviews"] / trade_tbl["reviews"]
ci = trade_tbl.apply(lambda r: wilson_interval(r["risk_reviews"], r["reviews"]), axis=1, result_type="expand")
trade_tbl["ci_lo"], trade_tbl["ci_hi"] = ci[0], ci[1]
display(trade_tbl.drop(columns="risk_reviews"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
trade_tbl["reviews"].sort_values().plot.barh(ax=axes[0], color="#6a8caf")
axes[0].set(title="Reviews by trade", xlabel="Reviews")
t = trade_tbl.sort_values("risk_rate")
axes[1].barh(t.index, t["risk_rate"], xerr=[t["risk_rate"] - t["ci_lo"], t["ci_hi"] - t["risk_rate"]], color="#d1495b", ecolor="#444", capsize=3)
axes[1].set(title="Share of 1–2★ reviews by trade (95% Wilson CI)", xlabel="Risk-signal rate")
plt.tight_layout()
plt.show()

hi, lo = trade_tbl["risk_rate"].idxmax(), trade_tbl["risk_rate"].idxmin()
display(Markdown(
    f"**Takeaway.** The share of 1–2★ reviews ranges from {trade_tbl.loc[lo, 'risk_rate']:.0%} (**{lo}**) to {trade_tbl.loc[hi, 'risk_rate']:.0%} (**{hi}**). "
    f"Trades differ, so provider comparisons in Layer 3 are made with the trade shown beside every score; small trades have wide intervals and should not be over-read."
))

In [ ]:
# ---- 4.4 What words separate complaints from praise? Weighted log-odds with an informative Dirichlet prior (Monroe et al., 2008) -
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer

# keep negations — "not", "never", "no" are among the most informative words in reviews
STOP = sorted(ENGLISH_STOP_WORDS - {"not", "no", "never", "nor", "cannot", "without"})
risk_txt = df.loc[df["review_stars"].isin(RISK_STARS), "text"]
safe_txt = df.loc[df["review_stars"].isin(SAFE_STARS), "text"]

cv = CountVectorizer(ngram_range=(1, 2), min_df=5, stop_words=STOP, lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
cv.fit(pd.concat([risk_txt, safe_txt]))
vocab = np.array(cv.get_feature_names_out())
ya = np.asarray(cv.transform(risk_txt).sum(axis=0)).ravel().astype(float)
yb = np.asarray(cv.transform(safe_txt).sum(axis=0)).ravel().astype(float)
prior = (ya + yb) / (ya + yb).sum() * 1000.0                      # prior mass proportional to overall frequency, total strength 1000
a0 = prior.sum()
delta = np.log((ya + prior) / (ya.sum() + a0 - ya - prior)) - np.log((yb + prior) / (yb.sum() + a0 - yb - prior))
z = delta / np.sqrt(1.0 / (ya + prior) + 1.0 / (yb + prior))
lo_df = pd.DataFrame({"term": vocab, "z": z, "risk_count": ya.astype(int), "safe_count": yb.astype(int)})
top_risk = lo_df.nlargest(15, "z")
top_safe = lo_df.nsmallest(15, "z")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top_risk["term"][::-1], top_risk["z"][::-1], color="#d1495b")
axes[0].set(title="Most characteristic of 1–2★ (risk) reviews", xlabel="log-odds z-score")
axes[1].barh(top_safe["term"][::-1], -top_safe["z"][::-1], color="#00798c")
axes[1].set(title="Most characteristic of 4–5★ reviews", xlabel="−log-odds z-score")
plt.tight_layout()
plt.show()
display(Markdown(
    "**Takeaway.** Highest-ranked terms for risk (1–2★) reviews: " + ", ".join(f"*{t}*" for t in top_risk["term"].head(8)) + ". "
    "Highest-ranked terms for 4–5★ reviews: " + ", ".join(f"*{t}*" for t in top_safe["term"].head(8)) + ". "
    "Read the left panel for words that describe *what happened* (behaviours, outcomes, money, repeat visits) as opposed to generic sentiment words — "
    "the more of those there are, the stronger the case for an **aspect layer** on top of sentiment: sentiment says *that* something went wrong, aspects say *what*."
))

In [ ]:
# ---- 4.5 Does the star rating over-simplify? Look at mixed (3★) and "high-star but critical" reviews -------------------------
def snippet(t, n=220):
    return t if len(t) <= n else t[: n - 1].rstrip() + "…"

rng = np.random.default_rng(SEED)
print("— A few 3★ (held out) reviews —")
for t in df.loc[df["review_stars"] == 3, "text"].sample(min(3, int((df["review_stars"] == 3).sum())), random_state=SEED):
    print(" •", snippet(t))
print("\n— A few 5★ reviews —")
for t in df.loc[df["review_stars"] == 5, "text"].sample(min(3, int((df["review_stars"] == 5).sum())), random_state=SEED):
    print(" •", snippet(t))

# Provider-level view: how much do star averages differ between providers, and how much does one provider's rating move over time?
prov_year = df.groupby(["business_id", "year"])["review_stars"].mean().unstack()
pm = df.groupby("business_id")["review_stars"].mean()
display(Markdown(
    f"**Takeaway.** Provider mean ratings span {pm.min():.1f}–{pm.max():.1f}★. A 3★ review often mixes praise and criticism in the same text, "
    f"which a single star value cannot represent; that is why 3★ is excluded from training and analysed separately in Section 5.7."
))

---
## 5. Layer 1 — Sentiment: Is This Review a Risk Signal?

**Task.** Binary text classification using star ratings as weak labels, exactly as in the proposal:

| Stars | Label | Meaning |
|---|---|---|
| 1–2 | **1 — risk signal** | the customer is reporting a problem |
| 4–5 | **0 — no immediate risk** | the customer is satisfied |
| 3 | *held out* | mixed sentiment; analysed separately (Section 5.7), never trained on |

**Three models of increasing capacity**, so we can quantify what each step buys:
1. **NLTK VADER** — a lexicon/rule baseline with *no training*. It tells us how far generic sentiment words alone go.
2. **TF-IDF + Logistic Regression** — a classical, interpretable supervised model (word and bigram features).
3. **Fine-tuned DistilBERT** — a transformer that reads word order and context ("did *not* finish", "would *never* hire").

**Evaluation protocol.** Stratified 70 / 15 / 15 train / validation / test split. The *validation* set is used for every tuning decision (hyper-parameters, decision
threshold, epoch); the *test* set is touched once, for the final report. We report **macro-F1 and weighted-F1, per-class precision and recall, and — most importantly —
negative-class (risk) recall**, because missing a serious complaint costs more than reviewing a false alarm. Each model is shown at two operating points:
the default threshold, and a threshold chosen on validation data to reach the recall target set in the configuration.

In [ ]:
# ---- 5.1 Labels and stratified splits ---------------------------------------------------------------------------------------------
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score, precision_recall_curve,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline

df["label"] = np.select([df["review_stars"].isin(RISK_STARS), df["review_stars"].isin(SAFE_STARS)], [1, 0], default=-1)
lab_idx = df.index[df["label"] >= 0].to_numpy()
mixed_idx = df.index[df["label"] < 0].to_numpy()

trainval_idx, test_idx = train_test_split(lab_idx, test_size=TEST_SIZE, stratify=df.loc[lab_idx, "label"], random_state=SEED)
train_idx, val_idx = train_test_split(trainval_idx, test_size=VAL_SIZE / (1 - TEST_SIZE), stratify=df.loc[trainval_idx, "label"], random_state=SEED)

split_tbl = pd.DataFrame({
    name: {"reviews": len(ix), "risk (1–2★)": int((df.loc[ix, "label"] == 1).sum()), "no risk (4–5★)": int((df.loc[ix, "label"] == 0).sum())}
    for name, ix in {"train": train_idx, "validation": val_idx, "test": test_idx, "held-out 3★": mixed_idx}.items()
}).T
split_tbl["risk share"] = split_tbl["risk (1–2★)"] / (split_tbl["risk (1–2★)"] + split_tbl["no risk (4–5★)"]).replace(0, np.nan)
display(split_tbl)

y = df["label"]
txt = df["text"]
assert not set(train_idx) & set(test_idx) and not set(val_idx) & set(test_idx), "leakage between splits"

In [ ]:
# ---- 5.2 Evaluation helpers (used by every model) -----------------------------------------------------------------------------------
def evaluate(y_true, y_pred, y_score, name, op):
    """One row of metrics. Class 1 = risk signal."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    return {
        "model": name, "operating point": op,
        "macro F1": f1_score(y_true, y_pred, average="macro"), "weighted F1": f1_score(y_true, y_pred, average="weighted"),
        "risk precision": p[1], "risk recall": r[1], "risk F1": f[1],
        "no-risk precision": p[0], "no-risk recall": r[0],
        "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score),
    }


def choose_threshold(y_true, score, min_recall=TARGET_RISK_RECALL, beta=F_BETA):
    """Business-driven threshold, chosen on VALIDATION data only.
    Take the highest-precision threshold whose risk recall still meets the target; if the target is unreachable, maximise F-beta."""
    prec, rec, thr = precision_recall_curve(y_true, score)
    prec, rec = prec[:-1], rec[:-1]
    ok = np.where(rec >= min_recall)[0]
    if len(ok):
        return float(thr[ok[np.argmax(prec[ok])]])
    fb = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-12)
    return float(thr[np.argmax(fb)])


def bootstrap_ci(y_true, y_pred, metric, n_boot=500, seed=SEED):
    """95% percentile bootstrap interval for a test-set metric."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    vals = []
    for _ in range(n_boot):
        ix = rng.integers(0, len(y_true), len(y_true))
        vals.append(metric(y_true[ix], y_pred[ix]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def risk_recall(yt, yp):
    return (yp[yt == 1] == 1).mean() if (yt == 1).any() else np.nan


model_scores = {}      # name -> {"val": score array, "test": score array, "thr_default": float, "thr_tuned": float}
results = []           # rows for the final comparison table


def register_model(name, val_score, test_score, thr_default):
    """Pick the recall-targeted threshold on validation, then evaluate both operating points once on the test set."""
    thr_tuned = choose_threshold(y.loc[val_idx], val_score)
    model_scores[name] = {"val": val_score, "test": test_score, "thr_default": thr_default, "thr_tuned": thr_tuned}
    for op, thr in (("default", thr_default), (f"recall target ≥{TARGET_RISK_RECALL:.2f}", thr_tuned)):
        results.append(evaluate(y.loc[test_idx], (test_score >= thr).astype(int), test_score, name, op))
    return thr_tuned

### 5.3 Model 1 — NLTK VADER (lexicon baseline)
VADER maps words to valence scores and applies rules for negation, intensifiers and punctuation. The review's *compound* score (−1 … +1) becomes a risk score
(`risk = −compound`). The conventional cut-off is ±0.05; a second operating point is tuned on validation data like every other model.

In [ ]:
def load_vader():
    """NLTK's VADER if its lexicon is available (downloaded on demand); otherwise the `vaderSentiment` package, which ships the identical lexicon."""
    try:
        import nltk
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        try:
            nltk.data.find("sentiment/vader_lexicon.zip")
        except LookupError:
            nltk.download("vader_lexicon", quiet=True)
        return SentimentIntensityAnalyzer(), "nltk"
    except Exception:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        return SentimentIntensityAnalyzer(), "vaderSentiment"


sia, vader_backend = load_vader()
print(f"VADER backend: {vader_backend}")
df["vader"] = txt.map(lambda t: sia.polarity_scores(t)["compound"])
vader_risk = -df["vader"]                                      # higher = riskier

# default rule: compound ≤ -0.05 → risk  (i.e. risk score ≥ 0.05)
register_model("VADER", vader_risk.loc[val_idx].to_numpy(), vader_risk.loc[test_idx].to_numpy(), thr_default=0.05)

# how separable are the classes by VADER alone?
display(df.loc[lab_idx].groupby("label")["vader"].describe()[["count", "mean", "25%", "50%", "75%"]].rename(index={0: "no risk (4–5★)", 1: "risk (1–2★)"}))

### 5.4 Model 2 — TF-IDF + Logistic Regression (supervised, interpretable)
* **Features:** TF-IDF of unigrams *and bigrams* (so "not happy", "never called" and "no show" become features), sub-linear term frequency, no stop-word removal (negations matter).
* **Imbalance:** `class_weight="balanced"` re-weights the loss instead of discarding data.
* **Tuning:** 5-fold stratified cross-validation on the *training* set only, scored on macro-F1; the validation set then sets the decision threshold.

In [ ]:
tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True, strip_accents="unicode", lowercase=True)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=3000, solver="liblinear")),
])
param_grid = {"tfidf__ngram_range": [(1, 1), (1, 2)], "clf__C": [0.5, 2, 8, 32]}
grid = GridSearchCV(tfidf_lr, param_grid, scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
t0 = time.time()
grid.fit(txt.loc[train_idx], y.loc[train_idx])
best_lr = grid.best_estimator_
print(f"best params: {grid.best_params_} | CV macro-F1 {grid.best_score_:.3f} | {time.time() - t0:.0f}s")

lr_val = best_lr.predict_proba(txt.loc[val_idx])[:, 1]
lr_test = best_lr.predict_proba(txt.loc[test_idx])[:, 1]
lr_thr = register_model("TF-IDF + LogReg", lr_val, lr_test, thr_default=0.5)
print(f"recall-targeted threshold chosen on validation data: {lr_thr:.3f}")

# Robustness: does the model generalise to providers it has never seen? (random split above can put one provider in train AND test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
g_tr, g_te = next(gss.split(lab_idx, groups=df.loc[lab_idx, "business_id"]))
grp_model = clone(best_lr).fit(txt.loc[lab_idx[g_tr]], y.loc[lab_idx[g_tr]])
grp_f1 = f1_score(y.loc[lab_idx[g_te]], grp_model.predict(txt.loc[lab_idx[g_te]]), average="macro")
rnd_f1 = f1_score(y.loc[test_idx], (lr_test >= 0.5).astype(int), average="macro")
display(Markdown(f"**Provider-held-out check:** macro-F1 on providers *never seen in training* = **{grp_f1:.3f}** versus **{rnd_f1:.3f}** on the random test split "
                 f"({'a small gap — no strong provider-specific leakage' if abs(grp_f1 - rnd_f1) < 0.03 else 'a noticeable gap — provider-specific wording may be inflating the random-split score'})."))

In [ ]:
# ---- What did the linear model learn? (interpretability — the reason to keep this model in the toolbox) ---------------------------
feat = np.array(best_lr.named_steps["tfidf"].get_feature_names_out())
coef = best_lr.named_steps["clf"].coef_.ravel()
order = np.argsort(coef)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_r, top_s = order[::-1][:15], order[:15]
axes[0].barh(feat[top_r][::-1], coef[top_r][::-1], color="#d1495b")
axes[0].set(title="Strongest features → RISK signal", xlabel="logistic-regression coefficient")
axes[1].barh(feat[top_s][::-1], coef[top_s][::-1], color="#00798c")
axes[1].set(title="Strongest features → NO immediate risk", xlabel="logistic-regression coefficient")
plt.tight_layout()
plt.show()

### 5.5 Model 3 — Fine-tuned DistilBERT
DistilBERT (a 40 % smaller, 60 % faster distillation of BERT that keeps ~97 % of its language-understanding quality) is fine-tuned end-to-end for the same binary task.
Design choices: a **class-weighted cross-entropy** loss (imbalance), **dynamic padding** (speed), **linear warm-up/decay** schedule, mixed precision on GPU, and
**model selection by validation PR-AUC** (threshold-independent). Predictions are cached to disk so re-running the notebook does not retrain.

> Needs a GPU (NVIDIA CUDA or Apple-silicon MPS) for practical speed. With `RUN_DISTILBERT = "auto"` this cell is skipped on CPU-only machines and the remaining sections are unaffected.

In [ ]:
def _accelerator() -> str:
    """'cuda' (NVIDIA), 'mps' (Apple silicon) or 'cpu'."""
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


USE_BERT = RUN_DISTILBERT is True or (RUN_DISTILBERT == "auto" and _accelerator() != "cpu")
bert_scores = None      # filled with {"val": …, "test": …, "mixed": …} when DistilBERT runs


def train_distilbert():
    """Fine-tune on the training split; return risk probabilities for validation, test and the held-out 3★ reviews."""
    import hashlib
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    tr_idx = train_idx if BERT_MAX_TRAIN is None else np.random.default_rng(SEED).choice(train_idx, min(BERT_MAX_TRAIN, len(train_idx)), replace=False)
    sig = hashlib.md5(json.dumps([BERT_CHECKPOINT, BERT_MAX_LEN, BERT_EPOCHS, BERT_BATCH, BERT_LR, int(len(tr_idx)), SEED, len(df)]).encode()).hexdigest()[:10]
    cache = CACHE_DIR / f"distilbert_{sig}.npz"
    if cache.exists():
        z = np.load(cache)
        print(f"loaded cached DistilBERT predictions ← {cache.name}")
        return {"val": z["val"], "test": z["test"], "mixed": z["mixed"]}

    torch.manual_seed(SEED)
    device = _accelerator()
    use_amp = device == "cuda"
    tok = AutoTokenizer.from_pretrained(BERT_CHECKPOINT)
    model = AutoModelForSequenceClassification.from_pretrained(BERT_CHECKPOINT, num_labels=2).to(device)

    def encode(idx, labelled=True):
        enc = tok(txt.loc[idx].tolist(), truncation=True, max_length=BERT_MAX_LEN)
        return [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "label": int(y.loc[ix]) if labelled else 0}
                for i, ix in enumerate(idx)]

    def collate(batch):
        padded = tok.pad([{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch], return_tensors="pt")
        padded["labels"] = torch.tensor([b["label"] for b in batch])
        return padded

    tr_feats, va_feats, te_feats = encode(tr_idx), encode(val_idx), encode(test_idx)
    mixed_feats = encode(mixed_idx, labelled=False) if len(mixed_idx) else []
    truncated = np.mean([len(f["input_ids"]) >= BERT_MAX_LEN for f in tr_feats])
    print(f"device={device} | train={len(tr_feats):,} | {truncated:.1%} of training reviews hit the {BERT_MAX_LEN}-token limit")

    counts = np.bincount([f["label"] for f in tr_feats], minlength=2)
    weights = torch.tensor(len(tr_feats) / (2.0 * np.maximum(counts, 1)), dtype=torch.float, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    loader = DataLoader(tr_feats, batch_size=BERT_BATCH, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=0.01)
    total_steps = max(1, len(loader) * BERT_EPOCHS)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    @torch.no_grad()
    def predict(feats):
        model.eval()
        out = []
        for batch in DataLoader(feats, batch_size=BERT_BATCH * 2, shuffle=False, collate_fn=collate):
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(**batch).logits
            out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
        return np.concatenate(out) if out else np.array([])

    best_ap, best_state = -1.0, None
    for epoch in range(1, BERT_EPOCHS + 1):
        model.train()
        running = 0.0
        for step, batch in enumerate(loader, 1):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
        val_p = predict(va_feats)
        ap = average_precision_score(y.loc[val_idx], val_p)
        print(f"epoch {epoch}/{BERT_EPOCHS} | train loss {running / len(loader):.4f} | val PR-AUC {ap:.4f}")
        if ap > best_ap:
            best_ap, best_state = ap, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    out = {"val": predict(va_feats), "test": predict(te_feats), "mixed": predict(mixed_feats)}
    np.savez(cache, **out)
    return out


if USE_BERT:
    t0 = time.time()
    bert_scores = train_distilbert()
    print(f"DistilBERT done in {time.time() - t0:.0f}s")
    bert_thr = register_model("DistilBERT", bert_scores["val"], bert_scores["test"], thr_default=0.5)
    print(f"recall-targeted threshold chosen on validation data: {bert_thr:.3f}")
else:
    print("DistilBERT skipped: no GPU detected (set HOMELENS_RUN_DISTILBERT=1 to force it on CPU, optionally with BERT_MAX_TRAIN to cap the training set).")

### 5.6 Model comparison and error analysis

In [ ]:
res = pd.DataFrame(results)
show_cols = ["model", "operating point", "macro F1", "weighted F1", "risk precision", "risk recall", "no-risk precision", "no-risk recall", "PR AUC"]
display(res[show_cols].style.format({c: "{:.3f}" for c in show_cols[2:]}).background_gradient(subset=["risk recall"], cmap="Greens").hide(axis="index"))

# Uncertainty: bootstrap 95% intervals on the test set at the recall-targeted operating point
ci_rows = []
for name, s in model_scores.items():
    pred = (s["test"] >= s["thr_tuned"]).astype(int)
    yt = y.loc[test_idx].to_numpy()
    lo_f, hi_f = bootstrap_ci(yt, pred, lambda a, b: f1_score(a, b, average="macro"))
    lo_r, hi_r = bootstrap_ci(yt, pred, risk_recall)
    ci_rows.append({"model": name, "macro F1 [95% CI]": f"{f1_score(yt, pred, average='macro'):.3f} [{lo_f:.3f}, {hi_f:.3f}]",
                    "risk recall [95% CI]": f"{risk_recall(yt, pred):.3f} [{lo_r:.3f}, {hi_r:.3f}]"})
display(pd.DataFrame(ci_rows))

fig, axes = plt.subplots(1, len(model_scores) + 1, figsize=(4.6 * (len(model_scores) + 1), 4.2))
for ax, (name, s) in zip(axes, model_scores.items()):
    cm = confusion_matrix(y.loc[test_idx], (s["test"] >= s["thr_tuned"]).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=["no risk", "risk"], yticklabels=["no risk", "risk"])
    ax.set(title=f"{name}\n(recall-targeted threshold)", xlabel="predicted", ylabel="actual")
ax = axes[-1]
for name, s in model_scores.items():
    pr, rc, _ = precision_recall_curve(y.loc[test_idx], s["test"])
    ax.plot(rc, pr, label=f"{name} (AP {average_precision_score(y.loc[test_idx], s['test']):.2f})")
ax.axvline(TARGET_RISK_RECALL, color="grey", ls=":", label=f"recall target {TARGET_RISK_RECALL:.2f}")
ax.set(title="Precision–recall (test)", xlabel="risk recall", ylabel="risk precision", ylim=(0, 1.02))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

tuned = res[res["operating point"] != "default"].set_index("model")
default = res[res["operating point"] == "default"].set_index("model")
best_name = tuned["macro F1"].idxmax()
lines = [f"**{m}**: macro-F1 {default.loc[m, 'macro F1']:.3f} → {tuned.loc[m, 'macro F1']:.3f} and risk recall {default.loc[m, 'risk recall']:.2f} → {tuned.loc[m, 'risk recall']:.2f} "
         f"(default → recall-targeted)" for m in model_scores]
display(Markdown("**Reading the table.** " + " · ".join(lines) + f". Best macro-F1 at the recall-targeted operating point: **{best_name}**. "
                 "Moving from the default to the recall-targeted threshold deliberately trades some precision for recall — the right trade when a missed serious complaint costs more than a false alarm."))

In [ ]:
# ---- Error analysis: where does the best classical model go wrong? -----------------------------------------------------------------
pred_lr = (lr_test >= lr_thr).astype(int)
err = df.loc[test_idx, ["review_stars", "text"]].assign(p_risk=lr_test, pred=pred_lr, actual=y.loc[test_idx].to_numpy())
fn = err[(err.actual == 1) & (err.pred == 0)].sort_values("p_risk")
fp = err[(err.actual == 0) & (err.pred == 1)].sort_values("p_risk", ascending=False)
print(f"TF-IDF + LogReg @ recall-targeted threshold: {len(fn)} missed risk reviews (false negatives), {len(fp)} false alarms (false positives)\n")
print("— Missed risk signals (lowest predicted risk first) —")
for r in fn.head(4).itertuples():
    print(f" • {r.review_stars}★ p={r.p_risk:.2f}: {snippet(r.text, 260)}")
print("\n— False alarms (highest predicted risk first) —")
for r in fp.head(4).itertuples():
    print(f" • {r.review_stars}★ p={r.p_risk:.2f}: {snippet(r.text, 260)}")
display(Markdown(
    "**How to read these.** Typical failure modes are (a) *mild or polite complaints* that use positive vocabulary, (b) *sarcasm*, (c) *mixed reviews* where a single "
    "bad episode sits in a mostly positive story, and (d) *star-rating noise* (a review whose text is positive but whose stars are low, or vice-versa). "
    "Cases (b)–(d) are largely label noise rather than model error, which is why Layer 2 shows the evidence sentence instead of trusting the score blindly."
))

### 5.7 The held-out 3★ reviews (RQ4)
3★ reviews were never used for training, so they are a natural test of whether the model recognises *mixed* sentiment: if it is behaving sensibly, their risk scores should sit
**between** the 1–2★ and 4–5★ distributions, with a meaningful share crossing the risk threshold.

In [ ]:
final_lr_mixed = best_lr.predict_proba(txt.loc[mixed_idx])[:, 1] if len(mixed_idx) else np.array([])
fig, ax = plt.subplots(figsize=(9, 4))
for lbl, name, c in ((1, "1–2★ (risk)", "#d1495b"), (0, "4–5★ (no risk)", "#00798c")):
    ax.hist(lr_test[(y.loc[test_idx] == lbl).to_numpy()], bins=25, alpha=0.55, color=c, label=name, density=True)
if len(final_lr_mixed):
    ax.hist(final_lr_mixed, bins=25, alpha=0.65, color="#edae49", label="3★ (held out)", density=True)
ax.axvline(lr_thr, color="k", ls="--", label=f"risk threshold {lr_thr:.2f}")
ax.set(title="Predicted risk probability (TF-IDF + LogReg)", xlabel="P(risk signal)", ylabel="density")
ax.legend()
plt.show()

if len(final_lr_mixed):
    share = (final_lr_mixed >= lr_thr).mean()
    display(Markdown(
        f"**Takeaway.** {share:.0%} of the {len(final_lr_mixed):,} three-star reviews cross the risk threshold (median P(risk) = {np.median(final_lr_mixed):.2f}, versus "
        f"{np.median(lr_test[(y.loc[test_idx] == 1).to_numpy()]):.2f} for 1–2★ and {np.median(lr_test[(y.loc[test_idx] == 0).to_numpy()]):.2f} for 4–5★). "
        f"A three-star review is therefore **not** automatically 'safe' or 'unsafe': these are the reviews where the aspect layer (Section 6) adds the most value, "
        f"because it shows *which* part of the experience was the problem."
        + (f" DistilBERT flags {(bert_scores['mixed'] >= model_scores['DistilBERT']['thr_tuned']).mean():.0%} of them." if bert_scores is not None else "")
    ))

### 5.8 Scoring every review for Layer 3
Layer 3 needs a risk probability for **every** review (including 3★) that is not contaminated by the model having seen the review during training. We therefore use
**5-fold out-of-fold (OOF) predictions** from the TF-IDF + LogReg model for all 1–2★ / 4–5★ reviews, and the final model fitted on all labelled data for the 3★ reviews.
We use the logistic-regression model here (rather than DistilBERT) because it can be cross-validated cheaply, is well calibrated, and is fully explainable.
A review is **flagged** when its probability reaches the recall-targeted threshold chosen on validation data.

In [ ]:
TAU = lr_thr                                               # decision threshold for "risk-flagged review"
oof = cross_val_predict(clone(best_lr), txt.loc[lab_idx], y.loc[lab_idx], cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        method="predict_proba", n_jobs=-1)[:, 1]
df["p_risk"] = np.nan
df.loc[lab_idx, "p_risk"] = oof
if len(mixed_idx):
    final_model = clone(best_lr).fit(txt.loc[lab_idx], y.loc[lab_idx])
    df.loc[mixed_idx, "p_risk"] = final_model.predict_proba(txt.loc[mixed_idx])[:, 1]
df["risk_flag"] = df["p_risk"] >= TAU

flag_by_star = df.groupby("review_stars")["risk_flag"].agg(["mean", "size"]).rename(columns={"mean": "share flagged", "size": "reviews"})
display(flag_by_star)
display(Markdown(f"**Sanity check.** Out-of-fold, {df.loc[df.label == 1, 'risk_flag'].mean():.0%} of 1–2★ reviews are flagged (≈ recall) and "
                 f"{df.loc[df.label == 0, 'risk_flag'].mean():.0%} of 4–5★ reviews are flagged (false-alarm rate). Overall {df['risk_flag'].mean():.0%} of all reviews are flagged."))

---
## 6. Layer 2 — Aspect Extraction: *What* Went Wrong, With Proof

Sentiment says *that* a customer is unhappy; it does not say *why*. Layer 2 tags each review with the **type of complaint** and returns the **evidence sentence** behind the tag, so
that every flag a manager sees can be checked in seconds.

### Why transparent rules rather than another neural model?
* **No labelled aspect data exists** for this domain; hand-labelling thousands of sentences is out of scope, while a rule system can be written, audited and corrected by the team.
* **Explainability is a requirement, not a nice-to-have.** Reviews are allegations; a manager must see the sentence that triggered a flag. Rules give that for free.
* **Cheap and deterministic** — it runs on a laptop in minutes, and a new phrase added to the lexicon takes effect immediately.

### Eight aspects (taxonomy from the proposal) and a severity weight
`workmanship · reliability · pricing · communication · timeliness · professionalism · warranty · safety`. Severity (0–1) reflects the cost of an *unaddressed* complaint
(safety highest) and is used only in Layer 3's provider score.

### Three spaCy components, each with a distinct job
| Component | Role here | Example |
|---|---|---|
| **`PhraseMatcher`** | Fast exact lookup of multi-word **lexicon cues**, case-insensitive (`attr="LOWER"`) | "hidden fees", "no show", "fire hazard" |
| **`Matcher`** | **Token-pattern rules** over lemmas / POS that capture *constructions* a word list cannot: negation + service verb, amounts versus quotes, durations of lateness | "*never* **called** back", "**$600 more** than quoted", "**three hours late**" |
| **`EntityRuler`** | Domain **named entities**: `HAZARD` (gas leak, carbon monoxide, exposed wiring …) and `TRADE` (plumber, roof, A/C …), written into `doc.ents` | `HAZARD` → safety signal; `TRADE` → context shown with the evidence |

### Avoiding false alarms: three gates
A bare keyword match is not a complaint ("*no* hidden fees", "they honoured the **warranty**"). Each cue therefore carries a tier:
* **strong** cues are unambiguous complaint language ("overcharged", "never showed up") and count on their own — unless **negated** just before ("*not* rude").
* **weak** cues are ambiguous words ("late", "dirty", "damaged") and count only in a sentence with **negative sentiment** (VADER compound ≤ `NEG_SENT_THRESH`) and when not negated.
* **topic** cues are neutral nouns ("warranty", "price", "permit") and count only in a **negative-sentiment sentence**.
* `Matcher` rules already encode a negative construction, so they count directly.
* **Layer 1 gates Layer 2.** Ambiguous cues (*weak*, *topic* and `HAZARD` entities) additionally require that Layer 1 flagged the **review** as a risk signal. A
  sentence like "they replaced it at no charge after the unit failed" can look negative to a sentence-level lexicon; the review-level model knows the review is positive.

In [ ]:
# ---- 6.1 The aspect lexicon (the single source of truth — it is also exported for the web app in Section 9) -----------------
ASPECTS = {
    "workmanship": {
        "label": "Workmanship / quality", "severity": 0.70,
        "description": "The work itself was poor, incomplete, damaged something, or did not fix the problem.",
        "strong": ["sloppy", "shoddy", "botched", "poor quality", "poorly done", "poor workmanship", "bad workmanship", "crooked", "cracked",
                   "still leaking", "leaks again", "leaked again", "not fixed", "redo", "redone", "re-do", "unfinished", "half done", "half-finished",
                   "made it worse", "worse than before", "cheap materials", "cut corners", "cutting corners", "faulty", "defective", "failed again",
                   "did not fix", "didn't fix", "never fixed", "poor work", "terrible work", "bad work", "uneven", "did a terrible job",
                   "did a poor job", "did a bad job", "wasn't done right", "not done right", "done incorrectly"],
        "weak": ["damaged", "broken", "ruined", "messy work", "mess", "problem persists", "problems persist"],
        "topic": ["quality", "workmanship"],
    },
    "reliability": {
        "label": "Reliability / no-shows", "severity": 0.60,
        "description": "Did not show up, abandoned the job, cancelled, or the problem kept coming back.",
        "strong": ["no show", "no-show", "never showed", "never showed up", "didn't show", "did not show", "never came", "never arrived", "stood me up",
                   "ghosted", "cancelled on me", "canceled on me", "cancelled last minute", "canceled last minute", "missed appointment",
                   "missed appointments", "missed two appointments", "left the job unfinished", "abandoned the job", "never came back", "did not come back",
                   "didn't come back", "never finished", "didn't finish", "did not finish", "walked off the job", "disappeared", "flaked",
                   "problem came back", "broke again", "broke down again"],
        "weak": ["cancelled", "canceled", "no call"],
        "topic": ["appointment", "appointments", "no one came", "nobody came"],
    },
    "pricing": {
        "label": "Pricing / billing", "severity": 0.60,
        "description": "Overcharging, hidden fees, price changes after the quote, upselling or billing disputes.",
        "strong": ["overcharge", "overcharged", "overcharging", "overpriced", "over priced", "hidden fees", "hidden fee", "hidden costs", "hidden charges",
                   "price gouging", "gouged", "rip off", "ripoff", "ripped off", "rip-off", "upsell", "upsold", "upselling", "bait and switch",
                   "bait & switch", "surprise charge", "surprise fee", "surprise fees", "extra charges", "extra charge", "way more than", "more than quoted",
                   "quote doubled", "price doubled", "exorbitant", "scam", "scammed", "double charged", "charged me twice", "unnecessary repairs",
                   "unnecessary work", "padded the bill", "inflated the price", "too expensive", "price increase", "raised the price"],
        "weak": ["expensive", "pricey", "inflated", "unnecessary", "didn't need", "did not need"],
        "negation_cancels_topic": True,                       # "at NO charge", "no fee" are praise, not complaints
        "topic": ["price", "prices", "quote", "estimate", "invoice", "bill", "charge", "charged", "fee", "fees", "cost", "costs", "deposit", "payment"],
    },
    "communication": {
        "label": "Communication", "severity": 0.40,
        "description": "Calls, messages and emails go unanswered; the customer is left uninformed.",
        "strong": ["never called back", "didn't call back", "did not call back", "never returned my call", "never returned my calls", "didn't return my call",
                   "didn't return my calls", "did not return my calls", "no response", "never responded", "didn't respond", "did not respond",
                   "no communication", "poor communication", "lack of communication", "terrible communication", "bad communication", "zero communication",
                   "unresponsive", "never answered", "didn't answer", "did not answer", "hung up on me", "no call back", "no callback",
                   "never got back to me", "didn't get back to me", "straight to voicemail", "no reply", "never replied", "left me hanging", "radio silence",
                   "never heard back", "didn't hear back", "hard to reach", "impossible to reach", "couldn't reach", "could not reach",
                   "never followed up", "no updates", "never updated"],
        "weak": [],
        "topic": ["communication", "voicemail", "voicemails", "phone calls", "callback"],
    },
    "timeliness": {
        "label": "Timeliness / delays", "severity": 0.30,
        "description": "Late arrival, missed deadlines, long waits, repeated rescheduling.",
        "strong": ["hours late", "days late", "weeks late", "showed up late", "arrived late", "running late", "took forever", "took too long", "took weeks",
                   "took months", "behind schedule", "kept rescheduling", "waited all day", "waiting all day", "waited for hours", "waited for weeks",
                   "never on time", "not on time", "overdue", "dragged on", "dragged out", "missed the deadline", "late again", "constant delays"],
        "weak": ["late", "delayed", "delay", "delays", "rescheduled", "slow", "long wait"],
        "topic": ["schedule", "scheduled", "on time", "punctual", "deadline", "waiting"],
    },
    "professionalism": {
        "label": "Professionalism / conduct", "severity": 0.40,
        "description": "Rude or dishonest behaviour, poor attitude, leaving the site dirty.",
        "strong": ["unprofessional", "rude", "disrespectful", "disrespect", "condescending", "arrogant", "dismissive", "yelled at", "yelling", "screamed at",
                   "cussed", "insulted", "left trash", "left garbage", "left a mess", "left a huge mess", "left debris", "tracked mud", "bad attitude",
                   "terrible attitude", "poor attitude", "hostile", "argumentative", "lied to me", "lied about", "dishonest", "liar", "shady", "sketchy",
                   "talked down to", "not courteous", "disrespected"],
        "weak": ["attitude", "dirty", "messy", "careless", "impatient", "lying"],
        "topic": ["professional", "professionalism", "courteous", "respectful"],
    },
    "warranty": {
        "label": "Warranty / follow-through", "severity": 0.70,
        "description": "Refused to honour a warranty or refund, or did not stand behind the work.",
        "strong": ["refused to honor", "refused to honour", "won't honor", "would not honor", "wouldn't honor", "did not honor", "didn't honor",
                   "denied my claim", "denied the claim", "won't stand behind", "wouldn't stand behind", "would not stand behind", "refused a refund",
                   "refused to refund", "no refund", "wouldn't refund", "would not refund", "won't refund", "void the warranty", "voided the warranty",
                   "voided my warranty", "not covered", "isn't covered", "not under warranty", "refused to fix", "refused to come back",
                   "won't come back to fix", "no follow up", "no follow-up", "didn't follow up", "did not follow up", "no guarantee",
                   "empty promises", "broken promise", "broke their promise", "broken promises"],
        "weak": [],
        "topic": ["warranty", "guarantee", "guaranteed", "refund", "reimburse", "reimbursement", "compensation", "dispute", "claim", "follow up", "follow-up"],
    },
    "safety": {
        "label": "Safety / licensing", "severity": 1.00,
        "description": "Hazards, code or permit violations, unlicensed or uninsured work, injury or property endangerment.",
        "strong": ["fire hazard", "fire risk", "started a fire", "caught fire", "sparking", "sparked", "electrocuted", "shocked me", "got shocked", "unsafe",
                   "dangerous", "hazardous", "not up to code", "code violation", "violated code", "failed inspection", "no permit", "without a permit",
                   "without permits", "unpermitted", "unlicensed", "not licensed", "no license", "uninsured", "not insured", "no insurance", "fell through",
                   "flooded", "flooding", "burst pipe", "injured", "injury", "hospital", "ambulance", "collapsed", "structural damage", "smelled gas",
                   "gas smell", "smell of gas", "leaking gas", "toxic", "fumes"],
        "weak": ["danger", "hazard", "unstable", "violation"],
        "topic": ["permit", "permits", "license", "licensed", "insurance", "insured", "inspection", "code"],
    },
}
ASPECT_ORDER = list(ASPECTS)
SEVERITY = {a: d["severity"] for a, d in ASPECTS.items()}

# EntityRuler vocabularies (domain entities written into doc.ents)
HAZARD_TERMS = ["gas leak", "gas leaks", "carbon monoxide", "exposed wire", "exposed wires", "exposed wiring", "electrical fire", "sparks", "water damage",
                "mold", "mildew", "asbestos", "live wire", "live wires", "short circuit", "tripped the breaker", "burning smell", "smoke"]
TRADE_TERMS = ["plumber", "plumbers", "plumbing", "electrician", "electricians", "wiring", "roofer", "roofers", "roof", "shingles", "hvac", "a/c", "ac unit",
               "air conditioning", "furnace", "water heater", "movers", "mover", "contractor", "contractors", "handyman", "landscaper", "landscaping",
               "cleaner", "cleaners", "pest control", "garage door", "flooring", "painter", "painters"]
ENTITY_PATTERNS = ([{"label": "HAZARD", "pattern": t} for t in HAZARD_TERMS] + [{"label": "TRADE", "pattern": t} for t in TRADE_TERMS])

print(f"{len(ASPECTS)} aspects | "
      f"{sum(len(d['strong']) + len(d['weak']) + len(d['topic']) for d in ASPECTS.values())} lexicon cues | {len(ENTITY_PATTERNS)} entity patterns")

In [ ]:
# ---- 6.2 Matcher rules: negation / construction patterns over lemmas and parts of speech --------------------------------------------
NEG = {"LOWER": {"IN": ["not", "never", "n't", "no", "nobody", "neither"]}}
ADV_OPT = {"POS": "ADV", "OP": "?"}                     # optional adverb: "never EVEN called", "did not really fix"
MONEY_WORD = {"LOWER": {"IN": ["more", "extra", "over", "higher", "above"]}}

MATCHER_RULES = {
    ("reliability", "never_showed"):    [[NEG, ADV_OPT, {"LEMMA": {"IN": ["show", "come", "finish", "complete"]}}]],
    ("reliability", "cancelled_on"):    [[{"LEMMA": "cancel"}, {"LOWER": {"IN": ["on", "last"]}}]],
    ("reliability", "left_unfinished"): [[{"LEMMA": {"IN": ["leave", "abandon"]}}, {"POS": "DET", "OP": "?"}, {"POS": "NOUN", "OP": "?"},
                                          {"LOWER": {"IN": ["unfinished", "incomplete", "half-done"]}}]],
    ("communication", "not_contacted"): [[NEG, ADV_OPT, {"LEMMA": {"IN": ["call", "respond", "answer", "reply", "text", "email", "contact",
                                                                         "communicate", "update", "notify", "inform", "explain"]}}]],
    ("communication", "not_returned"):  [[NEG, ADV_OPT, {"LEMMA": "return"}, {"LOWER": {"IN": ["my", "our", "any", "the"]}}, {"LEMMA": {"IN": ["call", "message", "email", "text"]}}]],
    ("communication", "never_got_back"):[[NEG, {"LOWER": "ever", "OP": "?"}, {"LEMMA": "get"}, {"LOWER": "back"}, {"LOWER": "to"}],
                                         [NEG, ADV_OPT, {"LEMMA": "hear"}, {"LOWER": "back"}]],
    ("communication", "ignored"):       [[{"LEMMA": "ignore"}, {"LOWER": {"IN": ["my", "our", "all", "every", "the"]}, "OP": "?"},
                                          {"LEMMA": {"IN": ["call", "email", "message", "text", "voicemail", "request", "question", "concern", "complaint"]}}]],
    ("pricing", "over_quote"):          [[{"LOWER": {"IN": ["more", "higher", "double", "triple", "twice"]}}, {"LOWER": "than"}, {"LOWER": "the", "OP": "?"},
                                          {"LEMMA": {"IN": ["quote", "estimate", "bid", "agree", "expect", "promise"]}}]],
    ("pricing", "charge_verbs"):        [[{"LEMMA": {"IN": ["overcharge", "upcharge", "overbill", "gouge", "upsell", "scam", "swindle"]}}],
                                         [{"LEMMA": "rip"}, {"LOWER": "off"}]],
    ("pricing", "amount_extra"):        [[{"LOWER": "$"}, {"LIKE_NUM": True}, MONEY_WORD],
                                         [{"LOWER": {"IN": ["extra", "additional"]}}, {"LOWER": "$"}, {"LIKE_NUM": True}],
                                         [{"LIKE_NUM": True}, {"LOWER": {"IN": ["dollars", "bucks"]}}, MONEY_WORD]],
    ("timeliness", "duration_late"):    [[{"LOWER": {"IN": ["minutes", "hours", "days", "weeks", "months", "hour", "day", "week", "month"]}},
                                          {"LOWER": {"IN": ["late", "behind", "overdue"]}}]],
    ("timeliness", "waited"):           [[{"LEMMA": "wait"}, {"LOWER": {"IN": ["all", "for", "over", "around"]}},
                                          {"LOWER": {"IN": ["day", "hours", "hour", "days", "weeks", "week", "months", "half", "two", "three", "four", "five",
                                                            "several", "many", "ever", "forever", "ages"]}}]],
    ("timeliness", "not_on_time"):      [[NEG, ADV_OPT, {"LOWER": "on"}, {"LOWER": "time"}],
                                         [NEG, ADV_OPT, {"LEMMA": {"IN": ["arrive", "start", "deliver"]}}, {"LOWER": {"IN": ["on", "by", "before"]}, "OP": "?"},
                                          {"LOWER": {"IN": ["time", "schedule", "deadline"]}}]],
    ("workmanship", "still_broken"):    [[{"LOWER": {"IN": ["still", "again"]}}, {"LEMMA": {"IN": ["leak", "break", "drip", "fail", "flood", "overheat", "malfunction", "crack"]}}],
                                         [{"LEMMA": {"IN": ["leak", "break", "drip", "fail", "flood", "overheat", "malfunction", "crack"]}}, {"LOWER": "again"}]],
    ("workmanship", "not_fixed"):       [[NEG, ADV_OPT, {"LEMMA": {"IN": ["fix", "repair", "solve", "resolve"]}}]],
    ("workmanship", "damaged_property"):[[{"LEMMA": {"IN": ["damage", "ruin", "destroy", "scratch", "break", "stain", "dent"]}}, {"LOWER": {"IN": ["my", "our", "the", "his"]}}]],
    ("professionalism", "yelled_at"):   [[{"LEMMA": {"IN": ["yell", "scream", "shout", "curse", "swear", "cuss"]}}, {"LOWER": "at"}]],
    ("professionalism", "no_apology"):  [[NEG, ADV_OPT, {"LEMMA": {"IN": ["apologize", "apologise", "listen", "care", "acknowledge"]}}]],
    ("professionalism", "hostile_verb"):[[{"LEMMA": {"IN": ["lie", "argue", "insult", "threaten"]}}]],
    ("warranty", "refused_to"):         [[{"LEMMA": {"IN": ["refuse", "decline", "deny", "reject"]}}, {"LOWER": "to", "OP": "?"}, {"LOWER": {"IN": ["my", "the", "our", "a", "any"]}, "OP": "?"},
                                          {"LEMMA": {"IN": ["honor", "honour", "cover", "refund", "fix", "repair", "reimburse", "warranty", "claim", "pay", "compensate", "replace"]}}],
                                         [NEG, ADV_OPT, {"LEMMA": "offer"}, {"LOWER": "to", "OP": "?"},
                                          {"LEMMA": {"IN": ["fix", "repair", "refund", "compensate", "replace", "pay", "help", "apologize"]}}]],
    ("warranty", "wont_cover"):         [[NEG, ADV_OPT, {"LEMMA": {"IN": ["honor", "honour", "cover", "stand", "refund", "reimburse"]}}]],
    ("safety", "no_permit"):            [[NEG, ADV_OPT, {"LEMMA": {"IN": ["pull", "get", "obtain", "have", "carry", "hold"]}}, {"LOWER": {"IN": ["a", "the", "any", "proper", "valid"]}, "OP": "?"},
                                          {"LEMMA": {"IN": ["permit", "license", "insurance"]}}]],
    ("safety", "injury_verb"):          [[{"LEMMA": {"IN": ["injure", "electrocute", "poison"]}}]],
}
print(f"{sum(len(v) for v in MATCHER_RULES.values())} token patterns in {len(MATCHER_RULES)} rules")

In [ ]:
# ---- 6.3 Build the spaCy pipeline: tokenizer → tagger → lemmatizer → fast sentence splitter → EntityRuler; + the two matchers ---------
import spacy
from spacy.matcher import Matcher, PhraseMatcher


def load_spacy_model(name="en_core_web_sm"):
    try:
        return spacy.load(name, disable=["parser", "ner"])
    except OSError:
        from spacy.cli import download
        download(name)
        return spacy.load(name, disable=["parser", "ner"])


nlp = load_spacy_model()
nlp.enable_pipe("senter")            # statistical sentence boundaries WITHOUT the (slow) dependency parser
ruler = nlp.add_pipe("entity_ruler", config={"phrase_matcher_attr": "LOWER", "overwrite_ents": True})
ruler.add_patterns(ENTITY_PATTERNS)

phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
for aspect, spec in ASPECTS.items():
    for tier in ("strong", "weak", "topic"):
        if spec[tier]:
            phrase_matcher.add(f"{aspect}|{tier}", [nlp.make_doc(t) for t in spec[tier]])

token_matcher = Matcher(nlp.vocab)
for (aspect, rule), patterns in MATCHER_RULES.items():
    token_matcher.add(f"{aspect}|{rule}", patterns)

print("pipeline:", nlp.pipe_names)

In [ ]:
# ---- 6.4 The extractor: hits → gates → one evidence record per (aspect, sentence) --------------------------------------------------
NEGATORS = {"not", "no", "never", "n't", "without", "nobody", "none", "hardly", "barely"}
SOURCE_PRIORITY = {"matcher": 3, "phrase-strong": 2, "entity": 1, "phrase-weak": 1, "phrase-topic": 0}


def extract_aspects(doc, review_id):
    """Return one record per (aspect, sentence) for a spaCy Doc. `complaint` is True when the hit passes the gates described above."""
    sent_cache = {}

    def sentence_of(i):
        s = doc[i].sent
        if s.start not in sent_cache:
            sent_cache[s.start] = (s, sia.polarity_scores(s.text)["compound"])
        return sent_cache[s.start]

    def negated(start):                       # a negator within the two tokens before the cue, inside the same sentence
        s = doc[start].sent
        return any(t.lower_ in NEGATORS for t in doc[max(s.start, start - 2):start])

    found = {}                                # (aspect, sentence start) → best record
    negated_cues = set()                      # (aspect, sentence start) where a complaint cue was explicitly negated ("no hidden fees")

    def keep(aspect, start, end, source, rule, complaint):
        s, comp = sentence_of(start)
        rec = {"review_id": review_id, "aspect": aspect, "source": source, "rule": rule, "cue": doc[start:end].text, "sentence": s.text.strip(),
               "sent_compound": comp, "complaint": complaint, "trade_context": ", ".join(sorted({e.text.lower() for e in s.ents if e.label_ == "TRADE"}))}
        key = (aspect, s.start)
        old = found.get(key)
        if old is None or (complaint, SOURCE_PRIORITY[source]) > (old["complaint"], SOURCE_PRIORITY[old["source"]]):
            found[key] = rec

    for mid, start, end in phrase_matcher(doc):                       # --- PhraseMatcher: lexicon cues
        aspect, tier = nlp.vocab.strings[mid].split("|")
        if tier in ("strong", "weak") and negated(start):
            negated_cues.add((aspect, doc[start].sent.start))
            continue                                                  # "not rude", "no hidden fees"
        if tier == "topic" and ASPECTS[aspect].get("negation_cancels_topic") and negated(start):
            continue                                                  # "no charge", "no fee"
        _, comp = sentence_of(start)
        keep(aspect, start, end, f"phrase-{tier}", "lexicon", complaint=(tier == "strong") or comp <= NEG_SENT_THRESH)

    for mid, start, end in token_matcher(doc):                        # --- Matcher: constructions (inherently negative)
        aspect, rule = nlp.vocab.strings[mid].split("|")
        keep(aspect, start, end, "matcher", rule, complaint=True)

    for ent in doc.ents:                                              # --- EntityRuler: hazards → safety (gated like a weak cue)
        if ent.label_ == "HAZARD" and not negated(ent.start):
            _, comp = sentence_of(ent.start)
            keep("safety", ent.start, ent.end, "entity", "HAZARD", complaint=comp <= NEG_SENT_THRESH)
    # a negated cue also silences the weaker cues around it ("No hidden fees" must not fire on the topic word "fees")
    return [r for k, r in found.items() if not (k in negated_cues and r["source"] in ("phrase-weak", "phrase-topic", "entity"))]


def analyze_text(text, review_id="adhoc"):
    """Ad-hoc helper for exploring a single review (also used by the unit tests below)."""
    return pd.DataFrame(extract_aspects(nlp(text), review_id))


analyze_text("They never called back and charged $600 more than quoted. The roof is still leaking again!")[["aspect", "source", "rule", "cue", "complaint", "sentence"]]

### 6.5 Unit tests for the rules
Rules are code and should be tested like code. Each row is a hand-written sentence with the aspects it **must** trigger and the ones it **must not** (negations, praise, neutral mentions).
Add a row whenever you fix a false positive or false negative — this table is the regression suite for the lexicon.

In [ ]:
RULE_TESTS = [
    # (sentence, aspects that MUST be flagged, aspects that MUST NOT be flagged)
    ("They never showed up and did not even call to cancel.",                   {"reliability"}, set()),
    ("Three hours late and the crew left early.",                               {"timeliness"}, {"reliability"}),
    ("They overcharged me, the bill was $600 more than quoted.",                {"pricing"}, set()),
    ("Hidden fees everywhere, it felt like a rip off.",                         {"pricing"}, set()),
    ("No hidden fees, the quote matched the invoice exactly.",                  set(), {"pricing"}),
    ("They never called back despite five voicemails.",                         {"communication"}, set()),
    ("Excellent communication, they called back within the hour.",              set(), {"communication"}),
    ("He was rude and dismissive when I asked a question.",                     {"professionalism"}, set()),
    ("They were not late at all, which was a nice change.",                     set(), {"timeliness"}),
    ("They refused to honor the warranty when it broke a week later.",          {"warranty"}, set()),
    ("They honored the warranty without any hassle, which I appreciated.",      set(), {"warranty"}),
    ("I smelled a gas leak after they left and it was dangerous.",              {"safety"}, set()),
    ("Everything was done to code, permitted, and the crew was licensed.",      set(), {"safety"}),
    ("The unlicensed crew did the work with no permit.",                        {"safety"}, set()),
    ("The pipe is still leaking and the repair did not fix the problem.",       {"workmanship"}, set()),
    ("Great work, the pipe looks fantastic and the price was fair.",            set(), {"workmanship", "pricing"}),
    ("They left the job unfinished and never came back.",                       {"reliability"}, set()),
    ("The technician waited all day for the parts and the project took forever.", {"timeliness"}, set()),
    ("Nobody ever got back to me after the estimate.",                         {"communication"}, set()),
    ("He never even apologized for the mess.",                                  {"professionalism"}, set()),
    ("They replaced the unit at no charge.",                                    set(), {"pricing"}),
    ("They refused to pay for the damage.",                                     {"warranty"}, set()),
    ("We have lived here for about six years.",                                 set(), set(ASPECT_ORDER)),
]


def run_rule_tests():
    rows = []
    for sent, must, must_not in RULE_TESTS:
        hits = analyze_text(sent)
        flagged = set(hits.loc[hits["complaint"], "aspect"]) if len(hits) else set()
        ok = must <= flagged and not (flagged & must_not)
        rows.append({"pass": "✅" if ok else "❌", "sentence": sent, "must flag": ", ".join(sorted(must)) or "—", "must not": ", ".join(sorted(must_not))[:40] or "—",
                     "flagged": ", ".join(sorted(flagged)) or "—"})
    return pd.DataFrame(rows)


rule_tests = run_rule_tests()
display(rule_tests)
n_pass = (rule_tests["pass"] == "✅").sum()
display(Markdown(f"**{n_pass} / {len(rule_tests)} rule tests pass.**" + ("" if n_pass == len(rule_tests) else " Fix the ❌ rows (edit the lexicon / Matcher rules above) before trusting Section 6.6.")))

### 6.6 Run the extractor on every review
All reviews are processed — not only the flagged ones — so that aspect prevalence can be compared across star groups (a validity check) and 3★ reviews can be analysed.
Results are cached on disk keyed by the lexicon and the data, so re-running the notebook is instant.

In [ ]:
import hashlib

sig = hashlib.md5((json.dumps([ASPECTS, {f"{k[0]}|{k[1]}": v for k, v in MATCHER_RULES.items()}, ENTITY_PATTERNS, NEG_SENT_THRESH], sort_keys=True, default=str)
                   + "".join(df["review_id"].astype(str))).encode()).hexdigest()[:12]
signals_cache = CACHE_DIR / f"aspect_signals_{sig}.csv"

if signals_cache.exists():
    signals = pd.read_csv(signals_cache, keep_default_na=False, dtype={"review_id": str})
    signals["complaint"] = signals["complaint"].astype(bool)
    print(f"loaded cached aspect signals ← {signals_cache.name}")
else:
    t0 = time.time()
    records = []
    for n, (doc, rid) in enumerate(zip(nlp.pipe(df["text"].tolist(), batch_size=64), df["review_id"].astype(str)), 1):
        records.extend(extract_aspects(doc, rid))
        if n % 5000 == 0:
            print(f"  {n:,} / {len(df):,} reviews processed ({time.time() - t0:.0f}s)")
    signals = pd.DataFrame(records, columns=["review_id", "aspect", "source", "rule", "cue", "sentence", "sent_compound", "complaint", "trade_context"])
    signals.to_csv(signals_cache, index=False)
    print(f"extracted in {time.time() - t0:.0f}s")

df["review_id"] = df["review_id"].astype(str)
# Layer 1 gates Layer 2: ambiguous cues (weak / topic / entity) only count inside reviews that Layer 1 flagged as a risk signal
signals["review_flagged"] = signals["review_id"].map(df.set_index("review_id")["risk_flag"]).fillna(False).astype(bool)
signals["signal"] = signals["complaint"] & (signals["source"].isin(["matcher", "phrase-strong"]) | signals["review_flagged"])
complaints = signals[signals["signal"]].copy()
print(f"{len(signals):,} cue hits → {len(complaints):,} pass the sentence and review gates | "
      f"{complaints['review_id'].nunique():,} of {len(df):,} reviews carry at least one complaint signal")

# one boolean column per aspect on the review table
for a in ASPECT_ORDER:
    df[f"has_{a}"] = df["review_id"].isin(complaints.loc[complaints["aspect"] == a, "review_id"])
df["n_aspects"] = df[[f"has_{a}" for a in ASPECT_ORDER]].sum(axis=1)
df["max_severity"] = df[[f"has_{a}" for a in ASPECT_ORDER]].mul(pd.Series(SEVERITY).reindex(ASPECT_ORDER).to_numpy()).max(axis=1)
complaints.groupby("aspect").agg(signals=("cue", "size"), reviews=("review_id", "nunique")).reindex(ASPECT_ORDER).fillna(0).astype(int)

In [ ]:
# ---- 6.7 Validity check #1 — do aspects fire where they should? Prevalence in risk (1–2★) vs satisfied (4–5★) reviews --------------
grp = df["review_stars"].map(lambda s: "risk" if s in RISK_STARS else "safe" if s in SAFE_STARS else "mixed")
prev = pd.DataFrame({g: df.loc[grp == g, [f"has_{a}" for a in ASPECT_ORDER]].mean().to_numpy() for g in ("risk", "mixed", "safe")}, index=ASPECT_ORDER)
prev["lift (risk ÷ safe)"] = prev["risk"] / prev["safe"].replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
prev[["risk", "mixed", "safe"]].plot.bar(ax=axes[0], color=["#d1495b", "#edae49", "#00798c"], width=0.8)
axes[0].set(title="Share of reviews with a complaint signal, by aspect", ylabel="share of reviews", xlabel="")
axes[0].tick_params(axis="x", rotation=35)
axes[0].legend(["1–2★", "3★", "4–5★"])
tot = df[[f"has_{a}" for a in ASPECT_ORDER]].sum()
axes[1].barh(ASPECT_ORDER[::-1], tot.to_numpy()[::-1], color="#6a8caf")
axes[1].set(title="Reviews with a complaint signal (all stars)", xlabel="reviews")
plt.tight_layout()
plt.show()
display(prev.style.format("{:.3f}").background_gradient(subset=["lift (risk ÷ safe)"], cmap="Greens"))

top_asp = prev["risk"].sort_values(ascending=False)
display(Markdown(
    f"**Takeaway.** In 1–2★ reviews the most frequent complaint types are **{', '.join(top_asp.index[:3])}** "
    f"({', '.join(f'{v:.0%}' for v in top_asp.values[:3])} of risk reviews). "
    f"Every aspect should show a lift above 1 (more common in risk reviews than in satisfied ones); "
    f"aspects with lift ≤ 1 ({', '.join(prev.index[(prev['lift (risk ÷ safe)'] <= 1).fillna(False)]) or 'none'}) need their cues reviewed — they are firing on praise or neutral text."
))

In [ ]:
# ---- 6.8 Validity check #2 — coverage. How many flagged reviews can we explain, and what are we missing? --------------------------
flagged = df[df["risk_flag"]]
explained = (flagged["n_aspects"] > 0).mean()
unexplained = flagged[flagged["n_aspects"] == 0]
print(f"{len(flagged):,} reviews flagged by Layer 1 → {explained:.0%} have at least one aspect with evidence; {len(unexplained):,} are 'unexplained'.")

if len(unexplained) >= 20:
    cv2 = CountVectorizer(ngram_range=(2, 3), min_df=3, stop_words=STOP, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
    Xu = cv2.fit_transform(unexplained["text"])
    top_unexp = pd.Series(np.asarray(Xu.sum(axis=0)).ravel(), index=cv2.get_feature_names_out()).sort_values(ascending=False).head(15)
    display(Markdown("**Most frequent phrases in unexplained risk reviews** — candidates for new lexicon cues (this is how the lexicon should grow):"))
    display(top_unexp.to_frame("count").T)

# a feel for the output: what a manager would see for three random flagged reviews
def highlight(sentence, cue):
    return re.sub(re.escape(cue), lambda m: f"**{m.group(0)}**", sentence, count=1, flags=re.I) if cue else sentence

for rid in flagged[flagged["n_aspects"] > 0].sample(min(3, int((flagged["n_aspects"] > 0).sum())), random_state=SEED)["review_id"]:
    row = df[df["review_id"] == rid].iloc[0]
    ev = complaints[complaints["review_id"] == rid].sort_values("sent_compound")
    block = f"**{row['review_stars']}★ · P(risk) = {row['p_risk']:.2f} · {row['trade']}**\n\n> {snippet(row['text'], 300)}\n\n"
    block += "\n".join(f"* **{e.aspect}** ({e.source}: *{e.cue}*) — “{highlight(snippet(e.sentence, 200), e.cue)}”" for e in ev.drop_duplicates('aspect').itertuples())
    display(Markdown(block))

In [ ]:
# ---- 6.9 Validity check #3 — human audit. Export a stratified sample; the team labels it; precision is computed here ---------------
ANNOT_DIR = DATA_DIR / "annotation"
ANNOT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_PATH = ANNOT_DIR / ("aspect_audit_DEMO.csv" if DEMO_MODE else "aspect_audit.csv")
AUDIT_PER_ASPECT = 15

if not AUDIT_PATH.exists():
    parts = [complaints[complaints["aspect"] == a].sample(min(AUDIT_PER_ASPECT, int((complaints["aspect"] == a).sum())), random_state=SEED) for a in ASPECT_ORDER]
    audit = pd.concat(parts)[["review_id", "aspect", "cue", "sentence"]].assign(correct="")
    audit.to_csv(AUDIT_PATH, index=False)
    print(f"Wrote a {len(audit)}-row audit sample → {AUDIT_PATH}\n"
          "TEAM TASK: open it, and for each row put 1 in `correct` if the sentence really is a complaint of that aspect, else 0. Two people should label independently. Re-run this cell.")
else:
    audit = pd.read_csv(AUDIT_PATH)
    labelled = audit[audit["correct"].isin([0, 1, "0", "1", 0.0, 1.0])].copy()
    labelled["correct"] = labelled["correct"].astype(float)
    if len(labelled) < 10:
        print(f"{AUDIT_PATH.name} exists but only {len(labelled)} rows are labelled — fill the `correct` column (1/0) and re-run.")
    else:
        prec_tbl = labelled.groupby("aspect")["correct"].agg(["mean", "sum", "size"]).rename(columns={"mean": "precision", "sum": "correct", "size": "audited"})
        ci2 = prec_tbl.apply(lambda r: wilson_interval(r["correct"], r["audited"]), axis=1, result_type="expand")
        prec_tbl["ci_lo"], prec_tbl["ci_hi"] = ci2[0], ci2[1]
        display(prec_tbl)
        overall = labelled["correct"].mean()
        display(Markdown(f"**Audit precision: {overall:.0%}** over {len(labelled)} labelled evidence sentences (target ≥ 80%). "
                         f"Weakest aspect: **{prec_tbl['precision'].idxmin()}** ({prec_tbl['precision'].min():.0%}) — start the next lexicon revision there."))

In [ ]:
# ---- 6.10 RQ4 — what do the mixed 3★ reviews complain about? --------------------------------------------------------------------
mixed_df = df[df["review_stars"].isin(MIXED_STARS)]
if len(mixed_df):
    mixed_prev = mixed_df[[f"has_{a}" for a in ASPECT_ORDER]].mean().rename("3★ reviews").to_frame()
    mixed_prev["1–2★ reviews"] = prev["risk"]
    mixed_prev["4–5★ reviews"] = prev["safe"]
    mixed_prev.index = [i.replace("has_", "") for i in mixed_prev.index]
    display(mixed_prev.style.format("{:.1%}").background_gradient(axis=None, cmap="Oranges"))
    both = mixed_df[(mixed_df["p_risk"] < TAU) & (mixed_df["n_aspects"] > 0)]
    display(Markdown(
        f"**Takeaway.** {(mixed_df['n_aspects'] > 0).mean():.0%} of 3★ reviews carry at least one complaint signal. "
        f"**{len(both):,}** three-star reviews carry an aspect complaint even though Layer 1 scores them *below* the risk threshold — "
        f"exactly the mixed cases where a single star value, and a sentiment score alone, hide a specific problem worth seeing."
    ))

---
## 7. Layer 3 — Provider Risk Intelligence: *Does It Keep Happening? What Should We Do?*

Individual flagged reviews are noise; **recurrence** is signal. Layer 3 rolls review-level evidence up to the provider and answers the questions a manager or homeowner actually has.

| Output | Definition |
|---|---|
| **Eligibility** | ≥ `MIN_REVIEWS_PER_PROVIDER` (20) reviews — below that, rates are too unstable to rank |
| **Negative rate** | share of the provider's reviews flagged by Layer 1 (out-of-fold probability ≥ τ), plus the Wilson *lower* bound so small samples are not over-ranked |
| **Recent vs prior** | negative rate in the last `RECENT_MONTHS` (24) months of data versus the 24 months before, with a Fisher exact test (`TREND_ALPHA`) → *worsening / stable / improving / insufficient data* |
| **Top recurring issues** | aspects found in ≥ `RECURRENCE_MIN` (3) *distinct* flagged reviews, each with its evidence sentences |
| **Risk score (0–100)** | 0.35·Wilson-LB negative rate + 0.25·recent negative rate + 0.25·severity + 0.15·recurrence. A **ranking aid, not a probability** |
| **Risk tier** | High / Medium / Low = top 15 % / next 25 % / rest of the eligible providers; any provider with ≥ 2 safety-signal reviews is lifted to at least Medium, ≥ 3 to High |
| **Recommended action** | rule-based from the top recurring issue, the safety escalation and the trend — separate wording for *managers* and *homeowners* |

> **Wording discipline.** Everything below is a *complaint signal detected in customer reviews*. It is not a finding that the provider did anything wrong.

In [ ]:
# ---- 7.1 Windows, per-provider profile, score, tier -----------------------------------------------------------------------------
SCORE_WEIGHTS = {"neg_rate_lb": 0.35, "recent_neg_rate": 0.25, "severity": 0.25, "recurrence": 0.15}
REF_DATE = df["date"].max()                                 # "today" for this dataset (it ends in early 2022, not in the present day)
RECENT_START = REF_DATE - pd.DateOffset(months=RECENT_MONTHS)
PRIOR_START = RECENT_START - pd.DateOffset(months=RECENT_MONTHS)
print(f"reference date {REF_DATE:%Y-%m-%d} | recent window ({RECENT_START:%Y-%m-%d}, {REF_DATE:%Y-%m-%d}] | prior window ({PRIOR_START:%Y-%m-%d}, {RECENT_START:%Y-%m-%d}]")


def profile_table(frame: pd.DataFrame, ref_date: pd.Timestamp) -> pd.DataFrame:
    """One row per provider: volume, negative rate, trend, recurring issues, severity, safety counts. `frame` must contain only data up to ref_date."""
    recent_start = ref_date - pd.DateOffset(months=RECENT_MONTHS)
    prior_start = recent_start - pd.DateOffset(months=RECENT_MONTHS)
    rows = []
    for bid, g in frame.groupby("business_id"):
        n, k = len(g), int(g["risk_flag"].sum())
        rec = g[(g["date"] > recent_start) & (g["date"] <= ref_date)]
        pri = g[(g["date"] > prior_start) & (g["date"] <= recent_start)]
        n_r, k_r, n_p, k_p = len(rec), int(rec["risk_flag"].sum()), len(pri), int(pri["risk_flag"].sum())

        if n_r >= MIN_WINDOW_REVIEWS and n_p >= MIN_WINDOW_REVIEWS:
            delta = k_r / n_r - k_p / n_p
            pval = stats.fisher_exact([[k_r, n_r - k_r], [k_p, n_p - k_p]])[1]
            direction = "worsening" if (delta > 0 and pval < TREND_ALPHA) else "improving" if (delta < 0 and pval < TREND_ALPHA) else "stable"
        else:
            delta, pval, direction = np.nan, np.nan, "insufficient data"

        flagged = g[g["risk_flag"]]
        counts = {a: int(flagged[f"has_{a}"].sum()) for a in ASPECT_ORDER}
        recurring = [a for a in ASPECT_ORDER if counts[a] >= RECURRENCE_MIN]
        rec_share = float(flagged[[f"has_{a}" for a in recurring]].any(axis=1).mean()) if (recurring and k) else 0.0
        rows.append({
            "business_id": bid, "n_reviews": n, "n_flagged": k, "neg_rate": k / n, "neg_rate_lb": wilson_interval(k, n)[0],
            "avg_stars": g["review_stars"].mean(),
            "n_recent": n_r, "recent_neg_rate": k_r / n_r if n_r >= MIN_WINDOW_REVIEWS else np.nan,
            "n_prior": n_p, "prior_neg_rate": k_p / n_p if n_p >= MIN_WINDOW_REVIEWS else np.nan,
            "trend_delta": delta, "trend_p": pval, "trend": direction,
            "severity": float(flagged["max_severity"].mean()) if k else 0.0, "recurrence_share": rec_share,
            "recurring_issues": recurring, "issue_counts": counts,
            "safety_reviews": int(g["has_safety"].sum()), "safety_recent": int(rec["has_safety"].sum()),
        })
    tbl = pd.DataFrame(rows)
    tbl["eligible"] = tbl["n_reviews"] >= MIN_REVIEWS_PER_PROVIDER
    return tbl


def add_scores(tbl: pd.DataFrame) -> pd.DataFrame:
    """Composite score and tier. Percentile cut-offs are computed among ELIGIBLE providers only."""
    w = SCORE_WEIGHTS
    recent_component = tbl["recent_neg_rate"].fillna(tbl["neg_rate"])               # fall back to the all-time rate when the window is too thin
    score = 100 * (w["neg_rate_lb"] * tbl["neg_rate_lb"] + w["recent_neg_rate"] * recent_component + w["severity"] * tbl["severity"] + w["recurrence"] * tbl["recurrence_share"])
    el = tbl["eligible"]
    cut_hi, cut_med = score[el].quantile(TIER_PCTS["High"]), score[el].quantile(TIER_PCTS["Medium"])
    tier = np.where(score >= cut_hi, "High", np.where(score >= cut_med, "Medium", "Low")).astype(object)
    tier = np.where((tbl["safety_reviews"] >= 2) & (tier == "Low"), "Medium", tier)
    tier = np.where(tbl["safety_reviews"] >= RECURRENCE_MIN, "High", tier)
    tbl = tbl.copy()
    tbl["safety_escalation"] = tbl["safety_reviews"] >= 2
    tbl["risk_score"] = np.where(el, score, np.nan)
    tbl["risk_tier"] = np.where(el, tier, "Insufficient data")
    tbl["rank"] = tbl["risk_score"].rank(ascending=False, method="min")
    return tbl


# Static provider attributes
prov_attr = (df.groupby("business_id").agg(name=("business_name", "first"), trade=("trade", "first"), city=("city", "first"), state=("state", "first"),
                                          categories=("categories", "first"), yelp_stars=("business_stars", "first")))
prov_attr["name"] = prov_attr["name"].fillna(pd.Series(prov_attr.index, index=prov_attr.index))

providers = add_scores(profile_table(df, REF_DATE)).merge(prov_attr, left_on="business_id", right_index=True)
elig = providers[providers["eligible"]].sort_values("rank")
print(f"{len(providers):,} providers → {len(elig):,} eligible (≥ {MIN_REVIEWS_PER_PROVIDER} reviews) | tiers: {elig['risk_tier'].value_counts().to_dict()}")

In [ ]:
# ---- 7.2 Actions: rule-based and deliberately separate for the two audiences -----------------------------------------------------
ACTION_MANAGER = {
    "safety": "ESCALATE: review the jobs behind the safety signals for permit, licence and hazard problems, contact the affected customers, and pause similar work until cleared.",
    "warranty": "Review the denied warranty / refund cases with the customers and confirm the warranty policy is being applied consistently.",
    "workmanship": "Audit recent jobs for rework; coach the crew, add a completion checklist and a quality call-back after each job.",
    "reliability": "Audit scheduling and dispatch for no-shows and abandoned jobs; add appointment confirmations and a backup-crew rule.",
    "pricing": "Compare quotes with final invoices; require a written change order before any extra charge.",
    "communication": "Set a call-back standard (e.g. same business day) and assign an owner to every unanswered message.",
    "timeliness": "Review scheduling buffers and arrival windows; send arrival-time notifications to customers.",
    "professionalism": "Coach the team on customer conduct and site clean-up; follow up personally on the flagged reviews.",
}
ACTION_HOMEOWNER = {
    "safety": "Before hiring, verify the licence, insurance and permit history; ask for proof for any gas, electrical or structural work.",
    "warranty": "Get warranty and refund terms in writing and ask how claims are handled before you sign.",
    "workmanship": "Ask for recent references and photos of similar jobs; agree what 'done' means (and a re-inspection) in writing.",
    "reliability": "Ask about scheduling and cancellation policy; avoid large deposits up front.",
    "pricing": "Get an itemised written quote and insist on written approval for any change in price.",
    "communication": "Test responsiveness with a call before committing; agree on a single point of contact.",
    "timeliness": "Agree a start and finish date in writing and ask what happens if it slips.",
    "professionalism": "Ask for references and agree on site-clean-up expectations in the contract.",
}


def recommend(row):
    """Return (manager action, homeowner action) for one provider row."""
    if not row["eligible"]:
        return ("Not enough reviews to rate — collect more feedback before acting.", "Not enough reviews to compare this provider reliably.")
    if row["safety_escalation"]:
        return (ACTION_MANAGER["safety"], ACTION_HOMEOWNER["safety"])
    recurring = sorted(row["recurring_issues"], key=lambda a: (-row["issue_counts"][a], -SEVERITY[a]))
    if recurring:
        a = recurring[0]
        return (ACTION_MANAGER[a], ACTION_HOMEOWNER[a])
    if row["trend"] == "worsening":
        return ("Negative rate is rising versus the prior period without one clear cause — read the latest flagged reviews and investigate recent changes (crews, suppliers, scheduling).",
                "Recent reviews are worse than earlier ones; read the most recent reviews before deciding.")
    return ("No recurring issue detected — keep routine monitoring.", "No recurring issue detected in reviews.")


providers[["action_manager", "action_homeowner"]] = providers.apply(lambda r: pd.Series(recommend(r)), axis=1)
elig = providers[providers["eligible"]].sort_values("rank")

In [ ]:
# ---- 7.3 Evidence: the sentences behind every recurring issue -------------------------------------------------------------------
review_meta = df[["review_id", "business_id", "date", "review_stars", "risk_flag"]]
ev_all = complaints.merge(review_meta, on="review_id", how="inner")


def provider_evidence(bid: str, aspect: str, k: int = 2):
    """The k most negative evidence sentences (one per review) for a provider's aspect, newest-first among ties."""
    e = ev_all[(ev_all["business_id"] == bid) & (ev_all["aspect"] == aspect) & ((ev_all["risk_flag"]) | (aspect == "safety"))]
    e = e.sort_values(["sent_compound", "date"], ascending=[True, False]).drop_duplicates("review_id").head(k)
    return [{"review_id": r.review_id, "date": f"{r.date:%Y-%m-%d}", "stars": int(r.review_stars), "cue": r.cue, "sentence": snippet(r.sentence, 300)} for r in e.itertuples()]


def provider_card(bid: str) -> str:
    p = providers[providers["business_id"] == bid].iloc[0]
    lines = [f"### {p['name']}  ·  {p['trade']}  ·  **{p['risk_tier']} risk**" + (f" (score {p['risk_score']:.0f}, rank {int(p['rank'])} of {len(elig)})" if p["eligible"] else ""),
             f"{p['n_reviews']} reviews · avg {p['avg_stars']:.1f}★ · negative-signal rate {p['neg_rate']:.0%} · trend: **{p['trend']}**"
             + (f" ({p['prior_neg_rate']:.0%} → {p['recent_neg_rate']:.0%})" if pd.notna(p["trend_delta"]) else "")]
    top = sorted([a for a in ASPECT_ORDER if p["issue_counts"][a] > 0], key=lambda a: (-p["issue_counts"][a], -SEVERITY[a]))[:3]
    for a in top:
        tag = "recurring" if a in p["recurring_issues"] else "isolated"
        lines.append(f"* **{ASPECTS[a]['label']}** — {p['issue_counts'][a]} flagged reviews ({tag})")
        for e in provider_evidence(bid, a, 1):
            lines.append(f"    * “{e['sentence']}” — {e['date']}, {e['stars']}★")
    lines.append(f"**Manager:** {p['action_manager']}  \n**Homeowner:** {p['action_homeowner']}")
    lines.append("*Complaint signals detected in customer reviews — unverified allegations, not findings.*")
    return "\n\n".join(lines)


display(Markdown("#### Top 10 providers by risk score"))
show = elig.head(10)[["rank", "name", "trade", "n_reviews", "avg_stars", "neg_rate", "trend", "risk_score", "risk_tier", "recurring_issues"]].copy()
show["recurring_issues"] = show["recurring_issues"].map(lambda r: ", ".join(r) or "—")
display(show.reset_index(drop=True).style.format({"rank": "{:.0f}", "avg_stars": "{:.1f}", "neg_rate": "{:.0%}", "risk_score": "{:.0f}"}).background_gradient(subset=["risk_score"], cmap="Reds"))

### 7.4 Does the score predict anything? A temporal backtest
A ranking is only useful if it anticipates the future. We **freeze** the data at the start of the recent window, build the profile and score using *only* the earlier reviews, and then check
whether providers scored as high-risk go on to receive a higher share of flagged reviews in the *following* period. The composite is compared with two naive alternatives a manager could use today:
the provider's historical star average and its historical negative rate.

In [ ]:
past = df[df["date"] <= RECENT_START]
bt = add_scores(profile_table(past, RECENT_START))
future = df[df["date"] > RECENT_START].groupby("business_id").agg(n_future=("risk_flag", "size"), future_neg=("risk_flag", "mean"))
bt = bt.merge(future, left_on="business_id", right_index=True)
bt = bt[bt["eligible"] & (bt["n_future"] >= MIN_WINDOW_REVIEWS)].copy()
print(f"backtest: {len(bt)} providers with ≥ {MIN_REVIEWS_PER_PROVIDER} reviews before {RECENT_START:%Y-%m-%d} and ≥ {MIN_WINDOW_REVIEWS} reviews after")

if len(bt) >= 8:
    cmp = []
    for name, series in (("HomeLens risk score", bt["risk_score"]), ("historical negative rate", bt["neg_rate"]), ("historical star average (inverted)", -bt["avg_stars"])):
        rho, p = stats.spearmanr(series, bt["future_neg"])
        cmp.append({"predictor": name, "Spearman ρ with future negative rate": rho, "p-value": p})
    display(pd.DataFrame(cmp).style.format({"Spearman ρ with future negative rate": "{:.2f}", "p-value": "{:.3f}"}).hide(axis="index"))
    tier_future = bt.groupby("risk_tier")["future_neg"].agg(["mean", "size"]).reindex(["High", "Medium", "Low"]).rename(columns={"mean": "future negative rate", "size": "providers"})
    display(tier_future.style.format({"future negative rate": "{:.1%}"}))
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    sns.regplot(data=bt, x="risk_score", y="future_neg", scatter_kws={"alpha": 0.6, "color": "#6a8caf"}, line_kws={"color": "#d1495b"}, ax=ax)
    ax.set(title="Score at the cut-off vs. negative rate afterwards", xlabel="risk score (history only)", ylabel="future negative-review rate")
    plt.tight_layout()
    plt.show()
    best_pred = pd.DataFrame(cmp).sort_values("Spearman ρ with future negative rate").iloc[-1]["predictor"]
    display(Markdown(f"**Takeaway.** The strongest predictor of the *future* negative rate was **{best_pred}**. "
                     f"High-tier providers went on to a future negative rate of {tier_future.loc['High', 'future negative rate']:.0%} versus {tier_future.loc['Low', 'future negative rate']:.0%} for Low-tier providers. "
                     f"With only {len(bt)} providers this is indicative rather than conclusive — see the p-values."))
else:
    print("Too few providers for a meaningful backtest on this data.")

In [ ]:
# ---- 7.5 RQ3 — does the score reveal risk that the star average hides? -----------------------------------------------------------
elig = providers[providers["eligible"]].sort_values("rank")
hidden = elig[(elig["avg_stars"] >= 4.0) & ((elig["risk_tier"].isin(["High", "Medium"])) | (elig["recurring_issues"].map(len) > 0))]
rho_all, _ = stats.spearmanr(elig["avg_stars"], elig["risk_score"])
worse = elig[elig["trend"] == "worsening"].sort_values("trend_delta", ascending=False)

display(Markdown(
    f"**Stars vs. risk.** Across {len(elig)} eligible providers the rank correlation between star average and risk score is ρ = {rho_all:.2f} — related, but far from identical. "
    f"**{len(hidden)}** providers ({len(hidden) / max(1, (elig['avg_stars'] >= 4.0).sum()):.0%} of those averaging ≥ 4★) still show a High/Medium tier or a recurring issue: "
    f"a star-sorted list would rank them as safe.  \n"
    f"**Trend.** {len(worse)} providers are significantly *worsening* (recent vs prior {RECENT_MONTHS} months) and "
    f"{(elig['trend'] == 'improving').sum()} improving; {(elig['trend'] == 'insufficient data').sum()} lack enough reviews in both windows to call."
))
if len(hidden):
    h = hidden.head(8)[["name", "trade", "n_reviews", "avg_stars", "risk_tier", "recurring_issues", "trend"]].copy()
    h["recurring_issues"] = h["recurring_issues"].map(lambda r: ", ".join(r) or "—")
    display(h.reset_index(drop=True).style.format({"avg_stars": "{:.1f}"}))
if len(worse):
    display(Markdown("**Largest significant deteriorations**"))
    display(worse.head(5)[["name", "trade", "n_recent", "n_prior", "prior_neg_rate", "recent_neg_rate", "trend_delta", "trend_p"]].reset_index(drop=True)
            .style.format({"prior_neg_rate": "{:.0%}", "recent_neg_rate": "{:.0%}", "trend_delta": "{:+.0%}", "trend_p": "{:.3f}"}))

---
## 8. Manager-Facing Risk Dashboard

The notebook's final analytical output: unstructured review text turned into a **prioritised operational decision**. The same data is exported in Section 9 to power the web dashboard
(`frontend/`); this static version is what the team presents.

In [ ]:
# ---- 8.1 Dashboard figure -------------------------------------------------------------------------------------------------------
TIER_COLORS = {"High": "#d1495b", "Medium": "#edae49", "Low": "#00798c"}
top15 = elig.head(15).copy()
label = lambda r: f"{snippet(str(r['name']), 28)}  ({r['trade'].split(' ')[0]})"

fig = plt.figure(figsize=(17, 11))
gs = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], hspace=0.35, wspace=0.28)

# (a) priority list
ax = fig.add_subplot(gs[0, 0])
ypos = np.arange(len(top15))[::-1]
ax.barh(ypos, top15["risk_score"], color=[TIER_COLORS[t] for t in top15["risk_tier"]])
ax.set_yticks(ypos, [label(r) for _, r in top15.iterrows()], fontsize=8)
for y_, (_, r) in zip(ypos, top15.iterrows()):
    main = max(r["issue_counts"], key=lambda a: (r["issue_counts"][a], SEVERITY[a])) if sum(r["issue_counts"].values()) else "—"
    ax.text(r["risk_score"] + 0.4, y_, f"{main}" + (" ⚠" if r["safety_escalation"] else "") + (" ↑" if r["trend"] == "worsening" else ""), va="center", fontsize=8)
ax.set(title="(a) Who needs attention first — top 15 by risk score  (⚠ safety · ↑ worsening)", xlabel="risk score (0–100)")
ax.set_xlim(0, top15["risk_score"].max() * 1.35)

# (b) issue heatmap
ax = fig.add_subplot(gs[0, 1])
heat = pd.DataFrame([r["issue_counts"] for _, r in top15.iterrows()], index=[snippet(str(n), 22) for n in top15["name"]])[ASPECT_ORDER]
sns.heatmap(heat, annot=True, fmt="d", cmap="Reds", cbar=False, ax=ax, linewidths=0.5, annot_kws={"fontsize": 8})
ax.set(title="(b) What is going wrong — flagged reviews per issue")
ax.tick_params(axis="x", rotation=40, labelsize=8)
ax.tick_params(axis="y", labelsize=8)

# (c) trend
ax = fig.add_subplot(gs[1, 0])
tr = elig.dropna(subset=["prior_neg_rate", "recent_neg_rate"])
if len(tr):
    ax.scatter(tr["prior_neg_rate"], tr["recent_neg_rate"], s=30 + 3 * tr["n_reviews"], c=[TIER_COLORS[t] for t in tr["risk_tier"]], alpha=0.75, edgecolor="white")
    lim = max(tr["prior_neg_rate"].max(), tr["recent_neg_rate"].max()) * 1.1 + 0.02
    ax.plot([0, lim], [0, lim], color="grey", ls="--", lw=1)
    ax.fill_between([0, lim], [0, lim], [lim, lim], color="#d1495b", alpha=0.05)
    for _, r in tr[tr["trend"] == "worsening"].nlargest(5, "trend_delta").iterrows():
        ax.annotate(snippet(str(r["name"]), 22), (r["prior_neg_rate"], r["recent_neg_rate"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set(xlim=(0, lim), ylim=(0, lim))
ax.set(title=f"(c) Getting better or worse? — negative rate, prior vs recent {RECENT_MONTHS} months", xlabel="prior-period negative rate", ylabel="recent negative rate")
ax.text(0.02, 0.95, "above the line = worsening", transform=ax.transAxes, fontsize=8, color="#d1495b", va="top")

# (d) stars hide risk
ax = fig.add_subplot(gs[1, 1])
ax.scatter(elig["avg_stars"], elig["risk_score"], c=[TIER_COLORS[t] for t in elig["risk_tier"]], s=35, alpha=0.75, edgecolor="white")
ax.axvline(4.0, color="grey", ls=":")
if len(hidden):
    ax.scatter(hidden["avg_stars"], hidden["risk_score"], s=130, facecolors="none", edgecolors="black", lw=1.2, label=f"≥ 4★ but flagged ({len(hidden)})")
    ax.legend(fontsize=8, loc="upper right")
ax.set(title="(d) Stars hide risk — star average vs risk score", xlabel="average review stars", ylabel="risk score")
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=t) for t, c in TIER_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, title="risk tier")
fig.suptitle("HomeLens AI — provider risk dashboard (complaint signals in customer reviews; unverified allegations, not findings)" + ("   [DEMO DATA]" if DEMO_MODE else ""), fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- 8.2 Drill-down: provider cards for the three highest-priority providers -----------------------------------------------------
for bid in elig.head(3)["business_id"]:
    display(Markdown(provider_card(bid)))

In [ ]:
# ---- 8.3 Key business insights (computed from this run) -------------------------------------------------------------------------
tier_n = elig["risk_tier"].value_counts()
asp_tot = pd.Series({a: int(df.loc[df["risk_flag"], f"has_{a}"].sum()) for a in ASPECT_ORDER}).sort_values(ascending=False)
trade_risk = elig.groupby("trade").agg(providers=("business_id", "size"), mean_score=("risk_score", "mean"), high=("risk_tier", lambda s: (s == "High").sum())).sort_values("mean_score", ascending=False)
best_model_row = res[res["operating point"] != "default"].sort_values("macro F1", ascending=False).iloc[0]
safety_prov = elig[elig["safety_escalation"]]

display(Markdown(f"""
#### What the analysis says
1. **The signal is in the text.** The best model (**{best_model_row['model']}**) separates risk from satisfied reviews with macro-F1 {best_model_row['macro F1']:.2f} and risk recall {best_model_row['risk recall']:.2f} at the recall-targeted operating point; the lexicon baseline reaches {res[(res.model == 'VADER') & (res['operating point'] != 'default')]['macro F1'].iloc[0]:.2f}.
2. **Complaints are specific.** Among flagged reviews the most common complaint types are **{', '.join(asp_tot.index[:3])}**; {explained:.0%} of flagged reviews carry at least one evidence sentence, so a manager gets *what* and *proof*, not just a score.
3. **Risk is concentrated.** Of {len(elig)} rated providers, {tier_n.get('High', 0)} are High tier, {tier_n.get('Medium', 0)} Medium and {tier_n.get('Low', 0)} Low; **{len(safety_prov)}** show repeated safety signals and need escalation.
4. **Stars hide the cause.** {len(hidden)} providers averaging ≥ 4★ still carry a recurring issue or an elevated tier, and {len(worse)} providers are significantly deteriorating — neither is visible in a star average.
5. **Trades differ.** Highest mean risk score: **{trade_risk.index[0]}** ({trade_risk['mean_score'].iloc[0]:.0f}); lowest: **{trade_risk.index[-1]}** ({trade_risk['mean_score'].iloc[-1]:.0f}) — compare providers within a trade.

#### What a stakeholder would do with it
* **Owner / operations manager:** open the High tier first; the recommended action is tied to the most frequent recurring issue; safety escalations go to the top regardless of rank.
* **Multi-location leadership:** watch the *worsening* list — it flags deterioration before the star average moves.
* **Reputation team:** reply first to reviews containing safety or warranty evidence; the rest can be acknowledged routinely.
* **Homeowner:** use the issue breakdown to pick the provider with the least recurring risk in the dimension that matters to them (price, punctuality, safety).
"""))

---
## 9. Export Artifacts for the Web App

The teammates building the **backend** (FastAPI) and **frontend** (Next.js) consume the JSON files written here — see `docs/DATA_CONTRACT.md` for the exact schema. The notebook is the
*offline* analytics pipeline; the web app is the *serving* layer, which is why heavyweight models (DistilBERT, spaCy) never need to run on Vercel.

In [ ]:
def jsonable(o):
    """Convert numpy / pandas scalars (and NaN) to plain JSON types."""
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if (isinstance(o, float) and math.isnan(o)) or (isinstance(o, np.floating) and np.isnan(o)) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (pd.Timestamp,)):
        return o.strftime("%Y-%m-%d")
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in o]
    if o is pd.NaT:
        return None
    return o


def yearly_history(bid):
    g = df[df["business_id"] == bid].groupby("year").agg(n=("risk_flag", "size"), neg_rate=("risk_flag", "mean"), avg_stars=("review_stars", "mean"))
    return [{"year": int(y_), "n": int(r.n), "neg_rate": round(float(r.neg_rate), 4), "avg_stars": round(float(r.avg_stars), 2)} for y_, r in g.iterrows()]


DISCLAIMER = "Complaint signals detected in customer reviews. Reviews are unverified customer allegations, not findings of wrongdoing."
meta = {
    "generated_at": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"), "data_mode": "sample" if DEMO_MODE else "yelp", "reference_date": f"{REF_DATE:%Y-%m-%d}",
    "recent_months": RECENT_MONTHS, "min_reviews": MIN_REVIEWS_PER_PROVIDER, "recurrence_min": RECURRENCE_MIN, "risk_threshold": round(float(TAU), 4),
    "scorer": "tfidf_logreg_oof", "score_weights": SCORE_WEIGHTS, "n_reviews": int(len(df)), "n_providers_total": int(len(providers)), "n_providers_eligible": int(len(elig)),
    "disclaimer": DISCLAIMER,
}

provider_records = []
for p in providers.sort_values(["eligible", "rank"], ascending=[False, True]).itertuples(index=False):
    p = p._asdict()
    issues = []
    for a in sorted([a for a in ASPECT_ORDER if p["issue_counts"][a] > 0], key=lambda a: (-p["issue_counts"][a], -SEVERITY[a])):
        issues.append({"aspect": a, "label": ASPECTS[a]["label"], "severity": SEVERITY[a], "n_reviews": p["issue_counts"][a],
                       "share_of_flagged": round(p["issue_counts"][a] / max(1, p["n_flagged"]), 4), "recurring": a in p["recurring_issues"],
                       "evidence": provider_evidence(p["business_id"], a, 2) if p["eligible"] else []})
    provider_records.append({
        "business_id": p["business_id"], "name": p["name"], "trade": p["trade"], "city": p["city"], "state": p["state"],
        "categories": split_categories(p["categories"]), "eligible": bool(p["eligible"]),
        "n_reviews": p["n_reviews"], "n_flagged": p["n_flagged"], "avg_stars": round(p["avg_stars"], 2), "yelp_stars": p["yelp_stars"],
        "neg_rate": round(p["neg_rate"], 4),
        "recent": {"n": p["n_recent"], "neg_rate": p["recent_neg_rate"]}, "prior": {"n": p["n_prior"], "neg_rate": p["prior_neg_rate"]},
        "trend": {"direction": p["trend"], "delta": p["trend_delta"], "p_value": p["trend_p"]},
        "risk_score": None if not p["eligible"] else round(p["risk_score"], 2), "risk_tier": p["risk_tier"], "rank": None if not p["eligible"] else int(p["rank"]),
        "safety_escalation": bool(p["safety_escalation"]), "issues": issues,
        "action_manager": p["action_manager"], "action_homeowner": p["action_homeowner"], "history": yearly_history(p["business_id"]),
    })

summary = {
    "meta": meta,
    "dataset": {"n_reviews": int(len(df)), "n_providers": int(df["business_id"].nunique()), "date_min": f"{df['date'].min():%Y-%m-%d}", "date_max": f"{df['date'].max():%Y-%m-%d}",
                "star_distribution": {int(k): int(v) for k, v in df["review_stars"].value_counts().sort_index().items()}, "median_words": float(df["n_words"].median())},
    "tiers": {k: int(v) for k, v in elig["risk_tier"].value_counts().items()},
    "trend_counts": {k: int(v) for k, v in elig["trend"].value_counts().items()},
    "models": [{k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in r.items()} for r in res.to_dict("records")],
    "aspects": [{"key": a, "label": ASPECTS[a]["label"], "severity": SEVERITY[a], "description": ASPECTS[a]["description"],
                 "share_in_risk_reviews": round(float(prev.loc[a, "risk"]), 4), "share_in_satisfied_reviews": round(float(prev.loc[a, "safe"]), 4),
                 "flagged_reviews": int(df.loc[df["risk_flag"], f"has_{a}"].sum())} for a in ASPECT_ORDER],
    "trades": [{"trade": t, "providers": int(r.providers), "reviews": int(r.reviews), "risk_rate": round(float(r.risk_rate), 4)} for t, r in trade_tbl.iterrows()],
    "disclaimer": DISCLAIMER,
}
lexicon = {"version": 1, "neg_sent_threshold": NEG_SENT_THRESH, "negators": sorted(NEGATORS),
           "aspects": {a: {k: v for k, v in spec.items()} for a, spec in ASPECTS.items()}}

for fname, payload in (("provider_risk.json", {"meta": meta, "providers": provider_records}), ("summary.json", summary), ("aspect_lexicon.json", lexicon)):
    path = OUTPUT_DIR / fname
    path.write_text(json.dumps(jsonable(payload), ensure_ascii=False, indent=1))
    print(f"wrote {path.relative_to(REPO_ROOT) if REPO_ROOT in path.parents else path}  ({path.stat().st_size / 1024:,.0f} KB)")
print("\nNext: `python scripts/sync_artifacts.py` copies these into backend/app/data/ (see docs/TEAM_GUIDE.md).")

---
## 10. Conclusions, Limitations and Next Steps

In [ ]:
display(Markdown(f"""
### Summary of results {'(DEMO DATA — replace with real-data numbers before submission)' if DEMO_MODE else ''}
* **Data.** {len(df):,} reviews of {df['business_id'].nunique():,} true home-service providers ({df['date'].min():%Y}–{df['date'].max():%Y}); {len(elig):,} providers have ≥ {MIN_REVIEWS_PER_PROVIDER} reviews and receive a risk profile.
* **Layer 1 – sentiment.** {', '.join(f"{m}: macro-F1 {tuned.loc[m, 'macro F1']:.2f} / risk recall {tuned.loc[m, 'risk recall']:.2f}" for m in model_scores)} (recall-targeted operating point, test set).
* **Layer 2 – aspects.** {len(ASPECTS)} aspects, {sum(len(d['strong']) + len(d['weak']) + len(d['topic']) for d in ASPECTS.values())} lexicon cues + {sum(len(v) for v in MATCHER_RULES.values())} token patterns + {len(ENTITY_PATTERNS)} entity patterns; {n_pass}/{len(rule_tests)} unit tests pass; {explained:.0%} of flagged reviews have an evidence sentence.
* **Layer 3 – providers.** {tier_n.get('High', 0)} High / {tier_n.get('Medium', 0)} Medium / {tier_n.get('Low', 0)} Low tier; {len(worse)} significantly worsening; {len(hidden)} providers look safe by stars but are not.
"""))

### Limitations (and what we would do about them)
| Limitation | Why it matters | Mitigation / next step |
|---|---|---|
| **Stars are weak labels.** A 2★ review may praise the work and complain about price; a 4★ review may hide a safety remark. | Label noise caps achievable F1 and can mislabel individual reviews. | Layer 2 shows the evidence sentence; next step: hand-label ≈500 reviews as a gold standard and measure label noise directly. |
| **Aspect rules have unknown recall.** The audit measures *precision* of what is flagged, not what is missed. | Some complaint phrasings will be absent from the lexicon. | The *unexplained-reviews* table (Section 6.8) lists missing phrases; compare against a zero-shot NLI or fine-tuned multi-label model on the gold set. |
| **Reviews are allegations, not facts.** | Reputational and legal risk if presented as findings. | Wording is always "complaint signal detected"; evidence is shown; a human reviews before any action. |
| **Selection bias.** Reviewers skew toward very happy and very unhappy customers; Yelp filters some reviews. | Negative rates over-state true failure rates. | Use scores for *relative* prioritisation within a trade, never as absolute failure rates. |
| **One city, one platform.** Tucson, AZ; Yelp only. | Vocabulary and base rates may not transfer. | Re-run on another city (the pipeline is parameterised by `CITY` / `STATE`). |
| **Random split, not temporal.** | Vocabulary drift could make test scores optimistic. | The temporal backtest (Section 7.4) checks the *provider score* on future data; next step: a time-based split for the classifier. |
| **Sparse windows.** Many providers have few reviews in each 24-month window. | Trend tests have low power. | Providers are labelled *insufficient data* rather than guessed. |
| **Score weights are judgement calls.** | Different weights change rankings. | Weights are explicit in `SCORE_WEIGHTS`; the backtest compares the composite with simple baselines. |

### Reproducibility
All randomness uses `SEED = 509`; split indices, model selection and bootstrap intervals are deterministic. Package versions used in this run are printed below.

In [ ]:
import importlib.metadata as md_
versions = {p: md_.version(p) for p in ("pandas", "numpy", "scikit-learn", "scipy", "spacy", "matplotlib", "seaborn", "vaderSentiment", "nltk", "torch", "transformers") if p in {d.metadata["Name"] for d in md_.distributions()}}
print(f"python {sys.version.split()[0]} | " + " | ".join(f"{k} {v}" for k, v in versions.items()))
print(f"seed {SEED} | data mode {DATA_MODE} | spaCy model {nlp.meta['name']} {nlp.meta['version']}")